# PHASE 11 — FULL BamiBERT · GIỮ NGUYÊN WIKI CRAWLER VÀ 8 BUCKET V3

- **Run All từ Colab:** mount Google Drive, unzip đúng `Phenikaa_Campus_Courier_2026_v3.zip`, đọc train/validation scenes riêng, load **chính xác** `bamibert_nlu_phase10_final_best.pt` bằng strict=True.
- **Wiki:** cell cào giữ nguyên từng hàm và cách print của phiên bản gốc. `WIKI_SLEEP=0.1`, `MAX_DEPTH=10`, `MAX_WORDS_PER_CLASS=3000`, **không tắt crawl**. Chỉ thay `WIKI_SEEDS`: 20 seed/cụm 2–3 tiếng, không có từ đơn `ăn`, `học`, `xe`. Bản Wiki đầy đủ có thể chạy lâu tuỳ mạng và số nhánh.
- **Sau crawl:** audit phụ không sửa code Wiki gốc; cách ly từ mơ hồ tránh `ăn chơi`/`ăn học` nhiễm vào BANK.
- **Generator V3:** giữ đúng 8 bucket `Aff/Neg × Acc/NoAcc × NoTypo/Typo`, **5.000/bucket = 40.000 mẫu V3 trước nhân 4 view**. Giữ nguyên hàm generator, audit và cách in cũ. Chỉ bổ sung template và từ vựng.
- **Bốn view cho mỗi mẫu train/validation:** nguyên văn; bỏ dấu; đúng một lỗi/thiếu ký tự; lỗi/thiếu ký tự + bỏ dấu. Validation thật từ `/validation/scenes.json` chỉ đánh giá, gộp thêm validation khó từ generator. Không nhúng JSON vào notebook.
- **Training:** model 8 head Phase 10; AdamW 2 nhóm learning rates, EMA, alignment, early stopping, in P5 theo format cũ, save best riêng.

**Lưu ý:** `Đang sinh 15.000` là đúng chuỗi print của cell cũ nhưng con số thực là 8 × 5.000 = 40.000. Không dùng `WIKI_RUN=False`; muốn chạy bộ code gốc đúng yêu cầu thì crawler cần mạng và có thể mất rất lâu.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# CELL 1 — Fresh Google Colab session: install, Drive, unzip the ACTUAL V3 dataset.
%pip -q install 'transformers>=4.44,<5' sentencepiece protobuf accelerate requests beautifulsoup4
import os, sys, re, json, copy, math, random, hashlib, gc, time, unicodedata, subprocess
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from tqdm.auto import tqdm

ZIP = '/content/drive/MyDrive/Phenikaa_Campus_Courier_2026_v3.zip'
ROOT = '/content/dataset/Phenikaa_Campus_Courier_2026_v3/delivery_public'
TRAIN_SCENES, VAL_SCENES = f'{ROOT}/train/scenes.json', f'{ROOT}/validation/scenes.json'
assert os.path.isfile(ZIP), f'Không tìm thấy ZIP trên Drive: {ZIP}'
# Equivalent to: !unzip -q -o "/content/drive/MyDrive/Phenikaa_Campus_Courier_2026_v3.zip" -d "/content/dataset"
if not (os.path.isfile(TRAIN_SCENES) and os.path.isfile(VAL_SCENES)):
    os.makedirs('/content/dataset', exist_ok=True)
    subprocess.run(['unzip','-q','-o',ZIP,'-d','/content/dataset'],check=True)
assert os.path.isfile(TRAIN_SCENES), f'Không tìm thấy {TRAIN_SCENES} sau giải nén'
assert os.path.isfile(VAL_SCENES), f'Không tìm thấy {VAL_SCENES} sau giải nén'

SEED=11092026
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MODEL_NAME='Qualcomm-AI-Research/BamiBERT'
MAX_LEN=256
SRC_DIR=Path('/content/drive/MyDrive/bamibert_nlu_phase11_v3_4views/bamibert_nlu_phase11_v3_4views_best.pt')
CHECKPOINT_PATH=''  # ONLY the user's requested .pt
OUT_DIR=Path('/content/drive/MyDrive/bamibert_nlu_phase12_v3_4views')
OUT_NAME='bamibert_nlu_phase11_v3_4views_best.pt'
OUT_DIR.mkdir(parents=True,exist_ok=True)

# Exactly 4 views for EVERY base sample in train, official validation and generated hard validation.
# V3: 8 bucket x 5000; bốn biến thể sẽ được tạo cho từng mẫu.
V3_PER_BUCKET=5000     # 8 V3 bucket nguyên gốc, 5.000 mỗi bucket
SPATIAL_TRAIN_BASE=8000
SPATIAL_HARD_BASE=900
V3_HARD_PER_BUCKET=250
ANCHOR_CONTRAST_BASE=400
BATCH_SIZE=32
EVAL_BATCH=64
P5_EPOCHS=50
P5_PATIENCE=30
W_VAL,W_PROBE=0.6,0.4
VAL_MAX_DROP=0.01
# WIKI gốc luôn chạy: MAX_DEPTH=10, MAX_WORDS_PER_CLASS=3000. Không dùng WIKI_RUN để bỏ qua crawl.
print('DEVICE',DEVICE,'TRAIN:',TRAIN_SCENES,'VAL:',VAL_SCENES,'CKPT:',CHECKPOINT_PATH)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 98.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 80.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
DEVICE cuda TRAIN: /content/dataset/Phenikaa_Campus_Courier_2026_v3/delivery_public/train/scenes.json VAL: /content/dataset/Phenikaa_Campus_Courier_2026_v3/delivery_public/validation/scenes.json CKPT: 


In [ ]:
# CELL 2 — Read the actual mission JSON directly from the unzipped Drive dataset.
# No embedded scenes, base64, fallback JSON, robot action labels, or validation-to-train mixing.
with open(TRAIN_SCENES, encoding='utf-8') as f: train_scenes=json.load(f)
with open(VAL_SCENES, encoding='utf-8') as f: val_scenes=json.load(f)
assert isinstance(train_scenes,list) and isinstance(val_scenes,list)
assert train_scenes and val_scenes
assert all('mission' in sc and 'text' in sc['mission'] for sc in train_scenes+val_scenes)
trids={sc['scene_id'] for sc in train_scenes}
vaids={sc['scene_id'] for sc in val_scenes}
assert not (trids & vaids), 'TRAIN/VAL scene_id bị trùng!'
print('REAL train scenes:',len(train_scenes),'| official validation scenes:',len(val_scenes))
print('TRAIN example:',train_scenes[0]['mission']['text'][:180])
print('VAL example:',val_scenes[0]['mission']['text'][:180])


REAL train scenes: 2000 | official validation scenes: 300
TRAIN example: Xin chao! Dung hnam voi phong doc nhe. Huy don giao nha xe. Thay vao do: Truoc khi chuyen bien ten khach den khong phai nha thi dau ma la cong truong gip minh, nho ghe lop hoc lay 
VAL example: Robot ơi, hôm qua đã giao ở khu ở của sinh viên rồi. Không giao thư viện trường nữa nhé, đổi lại: Trước khi vận chuyển bưu kiện tới địa điểm kề chỗ để xe nhất, nhớ ghé phòng thực n


In [ ]:
# PHASE 11 — 20 SEEDS ĐÃ KIỂM TRA / NHÓM; KHÔNG CÀO TỪ ĐƠN CHUNG CHUNG
import re, unicodedata, collections, json
PLACES = ['canteen','clinic','dorm','gate','lab','lecture','library','office','parking','sports']
MAP_KINDS = {'anchor_near','north_most','south_most','east_most','west_most'}
DIRECT_KINDS = ['none','near','far','north','south','east','west']
ALL_KINDS = DIRECT_KINDS + ['anchor_near','north_most','south_most','east_most','west_most']
HEADS_EXPECTED = {
 'goal':PLACES+['none'], 'via':['none']+PLACES,
 'gkind':ALL_KINDS, 'ganchor':['none']+PLACES,
 'vkind':ALL_KINDS,'vanchor':['none']+PLACES,
 'urgent':['no','yes'],'fragile':['no','yes']
}
H2I = {h:{v:i for i,v in enumerate(labels)} for h,labels in HEADS_EXPECTED.items()}
WIKI_SEEDS = {'canteen': ['căng tin', 'căn tin', 'nhà ăn', 'quán ăn', 'quán cơm', 'tiệm ăn', 'tiệm cơm', 'quầy cơm', 'quầy đồ ăn',
             'quầy thức ăn', 'khu ẩm thực', 'bếp ăn', 'phòng ăn', 'nhà bếp', 'quầy ăn uống', 'khu ăn uống',
             'quầy giải khát', 'quán ăn nhanh', 'khu dùng bữa', 'quầy suất ăn'],
 'clinic': ['phòng khám', 'trạm y tế', 'phòng y tế', 'trạm xá', 'bệnh xá', 'phòng sơ cứu', 'trạm sơ cứu', 'khu sơ cứu',
            'phòng cấp cứu', 'trạm cấp cứu', 'phòng bác sĩ', 'khu y tế', 'điểm y tế', 'phòng điều trị', 'khu điều trị',
            'trạm khám bệnh', 'nơi khám bệnh', 'khu khám bệnh', 'phòng khám bệnh', 'điểm sơ cứu'],
 'dorm': ['ký túc xá', 'kí túc xá', 'ký túc', 'kí túc', 'khu nội trú', 'nhà nội trú', 'tòa nội trú', 'dãy nội trú',
          'khối nội trú', 'phòng nội trú', 'khu lưu trú', 'nhà lưu trú', 'phòng lưu trú', 'khu ký túc', 'nhà ký túc',
          'tòa ký túc', 'dãy ký túc', 'khu KTX', 'tòa KTX', 'nhà KTX'],
 'gate': ['cổng trường', 'cổng chính', 'cổng phụ', 'cổng vào', 'cổng ra', 'cổng ra vào', 'cổng bảo vệ', 'chốt cổng',
          'chốt gác', 'trạm gác', 'chốt bảo vệ', 'trạm bảo vệ', 'lối vào', 'lối ra', 'lối ra vào', 'cửa vào', 'cửa ra',
          'cửa ra vào', 'cửa cổng', 'cửa chính'],
 'lab': ['phòng thí nghiệm', 'phòng thực nghiệm', 'phòng thực hành', 'phòng nghiên cứu', 'phòng lab', 'khu thí nghiệm',
         'khu thực nghiệm', 'khu thực hành', 'khu nghiên cứu', 'xưởng thực hành', 'xưởng thực nghiệm',
         'xưởng nghiên cứu', 'phòng chế tạo', 'xưởng chế tạo', 'xưởng cơ khí', 'phòng hóa nghiệm', 'phòng kiểm nghiệm',
         'khu kiểm nghiệm', 'trạm nghiên cứu', 'viện nghiên cứu'],
 'lecture': ['giảng đường', 'giảng phòng', 'phòng học', 'lớp học', 'nhà học', 'tòa học', 'khu giảng đường',
             'dãy giảng đường', 'tòa giảng đường', 'phòng giảng', 'phòng giảng dạy', 'khu lớp học', 'dãy lớp học',
             'dãy phòng học', 'khu nhà học', 'tòa nhà học', 'phòng lý thuyết', 'lớp lý thuyết', 'phòng học chung',
             'phòng chuyên đề'],
 'library': ['thư viện', 'thư khố', 'kho sách', 'phòng đọc', 'phòng đọc sách', 'khu đọc sách', 'quầy mượn sách',
             'quầy trả sách', 'phòng mượn sách', 'phòng tham khảo', 'kho tài liệu', 'phòng tài liệu', 'kho học liệu',
             'khu học liệu', 'quầy thư viện', 'phòng thư viện', 'nhà thư viện', 'khu thư viện', 'tòa thư viện',
             'phòng tra cứu'],
 'office': ['văn phòng', 'tòa hiệu bộ', 'nhà hiệu bộ', 'khu hiệu bộ', 'phòng hành chính', 'khu hành chính',
            'tòa hành chính', 'phòng giáo vụ', 'phòng đào tạo', 'văn phòng khoa', 'văn phòng trường', 'phòng học vụ',
            'ban giám hiệu', 'phòng giám hiệu', 'phòng điều hành', 'phòng quản lý', 'phòng tiếp nhận',
            'quầy hành chính', 'phòng công vụ', 'ban quản lý'],
 'parking': ['bãi xe', 'nhà xe', 'hầm xe', 'bãi đỗ xe', 'bãi đậu xe', 'bãi gửi xe', 'bãi giữ xe', 'nhà để xe',
             'nhà giữ xe', 'hầm để xe', 'hầm gửi xe', 'hầm đỗ xe', 'khu để xe', 'khu gửi xe', 'điểm gửi xe',
             'chỗ đỗ xe', 'chỗ để xe', 'chỗ gửi xe', 'bãi xe máy', 'bãi ô tô'],
 'sports': ['nhà thi đấu', 'nhà thể chất', 'khu thể thao', 'sân thể thao', 'sân bóng', 'sân tập', 'sân thi đấu',
            'khu thi đấu', 'sân bóng đá', 'sân bóng rổ', 'sân cầu lông', 'sân quần vợt', 'phòng gym', 'phòng tập',
            'khu tập luyện', 'khu thể chất', 'nhà luyện tập', 'nhà tập luyện', 'sân vận động', 'sân tennis'],
 'near': ['gần nhất', 'liền kề', 'ngay cạnh', 'sát cạnh', 'kề bên', 'sát bên', 'ngay sát', 'bên cạnh', 'cạnh bên',
          'gần kề', 'kề cận', 'sát vách', 'ở gần', 'ở sát', 'ở cạnh', 'kề cạnh', 'gần sát', 'tiếp giáp', 'lân cận',
          'ngay kế'],
 'far': ['xa nhất', 'cách xa', 'ở xa', 'khá xa', 'rất xa', 'cực xa', 'xa tít', 'xa lắc', 'xa xôi', 'xa hẳn', 'xa vời',
         'xa cách', 'xa lơ', 'xa ngái', 'tít tắp', 'tít mù', 'cách biệt', 'cách khá xa', 'xa thật xa', 'xa tít tắp'],
 'north': ['phía bắc', 'hướng bắc', 'mạn bắc', 'mé bắc', 'bên bắc', 'khu bắc', 'vùng bắc', 'phía trên', 'bên trên',
           'hướng trên', 'mé trên', 'đầu trên', 'nửa trên', 'theo hướng bắc', 'ở phía bắc', 'nằm phía bắc',
           'nằm bên trên', 'thuộc phía bắc', 'theo phía bắc', 'tại mạn bắc'],
 'south': ['phía nam', 'hướng nam', 'mạn nam', 'mé nam', 'bên nam', 'khu nam', 'vùng nam', 'phía dưới', 'bên dưới',
           'hướng dưới', 'mé dưới', 'đầu dưới', 'nửa dưới', 'theo hướng nam', 'ở phía nam', 'nằm phía nam',
           'nằm bên dưới', 'thuộc phía nam', 'theo phía nam', 'tại mạn nam'],
 'east': ['phía đông', 'hướng đông', 'mạn đông', 'mé đông', 'bên đông', 'khu đông', 'vùng đông', 'phía phải',
          'bên phải', 'hướng phải', 'mé phải', 'cạnh phải', 'nửa phải', 'theo hướng đông', 'ở phía đông',
          'nằm phía đông', 'nằm bên phải', 'thuộc phía đông', 'theo phía đông', 'tại mạn đông'],
 'west': ['phía tây', 'hướng tây', 'mạn tây', 'mé tây', 'bên tây', 'khu tây', 'vùng tây', 'phía trái', 'bên trái',
          'hướng trái', 'mé trái', 'cạnh trái', 'nửa trái', 'theo hướng tây', 'ở phía tây', 'nằm phía tây',
          'nằm bên trái', 'thuộc phía tây', 'theo phía tây', 'tại mạn tây'],
 'urgent': ['khẩn cấp', 'cấp bách', 'khẩn thiết', 'khẩn trương', 'hỏa tốc', 'tốc hành', 'cấp tốc', 'tức tốc', 'gấp rút',
            'gấp gáp', 'cần gấp', 'giao gấp', 'giao ngay', 'làm ngay', 'gửi ngay', 'chuyển gấp', 'sát giờ', 'phải gấp',
            'sắp muộn', 'ưu tiên cao'],
 'fragile': ['dễ vỡ', 'dễ bể', 'dễ nứt', 'dễ gãy', 'dễ hỏng', 'dễ sứt', 'dễ móp', 'dễ trầy', 'dễ xước', 'mong manh',
             'thủy tinh', 'đồ gốm', 'đồ sứ', 'đồ kính', 'hàng kính', 'hàng sứ', 'đồ dễ vỡ', 'hàng dễ vỡ',
             'tránh va đập', 'nhẹ tay']}
# 20 curated query seeds / category (2–3 syllables for exact Vietnamese names like 'ký túc xá').
# Wiki results are *not* auto-imported into BANK; only listed vetted seed expressions are used.
BANK = {k:list(v) for k,v in WIKI_SEEDS.items()}
# Tên thực tế phổ biến, được phê duyệt thủ công, KHÔNG phải từ Wiki tự tìm.
BANK['dorm'] += ['KTX']
BANK['gate'] += ['cổng']
BANK['library'] += ['thư viện trường']
BANK['office'] += ['hiệu bộ']
BANK['sports'] += ['sân bóng chuyền']
BANK['lab'] += ['phòng thực nghiệm']
BANK['fragile'] += ['dễ vỡ','dễ bể']

def vocab_fold(s):
 t=unicodedata.normalize('NFD',str(s).lower().replace('đ','d'))
 return ''.join(c for c in t if unicodedata.category(c)!='Mn')
def vocab_key(s):
 return re.sub(r'[^a-z0-9]+',' ',vocab_fold(s)).strip()
def fold(s):return vocab_fold(s)

def check_wiki_seeds():
 assert set(HEADS_EXPECTED)=={'goal','via','gkind','ganchor','vkind','vanchor','urgent','fragile'}
 owners=collections.defaultdict(set)
 for k,ss in WIKI_SEEDS.items():
  assert len(ss)==20,(k,len(ss))
  assert len({vocab_key(s) for s in ss})==20,('dup',k)
  for s in ss:
   assert 2<=len(s.split())<=3 and 'sinh vien' not in vocab_key(s),(k,s)
   assert vocab_key(s) not in {'an','hoc','xe','nha','phong','doc','choi','di','diem','khu','sach','o'}, (k,s)
   if k in PLACES:owners[vocab_key(s)].add(k)
 assert not {s:x for s,x in owners.items() if len(x)>1},'Cross-class seed'
 print('OK: 20 curated Wiki seeds ×',len(WIKI_SEEDS),'groups =',sum(len(v) for v in WIKI_SEEDS.values()))
check_wiki_seeds()
ITEMS = ['hộp giấy','bưu kiện','linh kiện điện tử','tập tài liệu','túi đồ','bộ vợt',
 'bộ sách','hộp thuốc','túi sơ cứu','biển tên','thẻ giữ xe','giấy tờ',
 'thùng hàng','hồ sơ','gói hàng','khay cơm','đồ gốm','đơn hàng','quả bóng']
KINDS = DIRECT_KINDS[1:]  # V3 gốc chỉ sinh referent trực tiếp, spatial dùng extension riêng.


OK: 20 curated Wiki seeds × 18 groups = 360


In [ ]:
# CELL 4 — Ground the 8 semantic heads from EACH real mission, without using robot action labels.
def from_scene(scene, origin):
    m=scene['mission']; g=m.get('goal_ref') or {}; v=m.get('via_ref') or {}
    gkind=g.get('kind') or 'none'; vkind=v.get('kind') or 'none'
    ganchor=g.get('anchor') or 'none'; vanchor=v.get('anchor') or 'none'
    gp='none' if gkind in MAP_KINDS else m['goal']
    vp='none' if vkind in MAP_KINDS else (m.get('via') or 'none')
    labels={'goal':gp,'via':vp,'gkind':gkind,'ganchor':ganchor,
            'vkind':vkind,'vanchor':vanchor,
            'urgent':'yes' if m.get('urgent',False) else 'no',
            'fragile':'yes' if m.get('fragile',False) else 'no'}
    assert set(labels)==set(HEADS_EXPECTED)
    assert all(v in H2I[h] for h,v in labels.items()),(labels,m.get('text'))
    for k,a in [(gkind,ganchor),(vkind,vanchor)]:
        assert (a in PLACES) if k in ('near','far','anchor_near') else (a=='none'),(k,a)
    assert (gp=='none')==(gkind in MAP_KINDS)
    if vp=='none' and vkind not in MAP_KINDS:assert vkind=='none'
    return {'text':m['text'].strip(), 'y':{h:H2I[h][v] for h,v in labels.items()},
            'origin':origin, 'scene_id':scene['scene_id']}

train_real=[from_scene(sc,'official_train') for sc in train_scenes]
official_val=[from_scene(sc,'official_val') for sc in val_scenes]
assert not ({vocab_key(it['text']) for it in train_real} &
            {vocab_key(it['text']) for it in official_val}), 'train/val mission overlap'
print('TRAIN real:',len(train_real), '| OFFICIAL VAL:',len(official_val))
print('Train goal_ref kinds:',Counter(HEADS_EXPECTED['gkind'][r['y']['gkind']] for r in train_real))


TRAIN real: 2000 | OFFICIAL VAL: 300
Train goal_ref kinds: Counter({'none': 837, 'anchor_near': 723, 'west_most': 89, 'east_most': 81, 'south_most': 81, 'north_most': 79, 'near': 48, 'far': 29, 'north': 10, 'south': 9, 'east': 8, 'west': 6})


In [ ]:
# ================================================================

# CELL: WIKTIONARY FULL SYNONYM CRAWLER (PHIÊN BẢN CẢI TIẾN)

# ================================================================

import requests

import urllib.parse

import re

import time

from bs4 import BeautifulSoup

from collections import defaultdict



WIKI_SLEEP = 0.1

MAX_DEPTH = 10

MAX_WORDS_PER_CLASS = 3000

REQUEST_TIMEOUT = 10

VERBOSE = True



WIKI_SEEDS = {'canteen': ['căng tin',
             'căn tin',
             'nhà ăn',
             'quán ăn',
             'quán cơm',
             'tiệm ăn',
             'tiệm cơm',
             'quầy cơm',
             'quầy đồ ăn',
             'quầy thức ăn',
             'khu ẩm thực',
             'bếp ăn',
             'phòng ăn',
             'nhà bếp',
             'quầy ăn uống',
             'khu ăn uống',
             'quầy giải khát',
             'quán ăn nhanh',
             'khu dùng bữa',
             'quầy suất ăn'],
 'clinic': ['phòng khám',
            'trạm y tế',
            'phòng y tế',
            'trạm xá',
            'bệnh xá',
            'phòng sơ cứu',
            'trạm sơ cứu',
            'khu sơ cứu',
            'phòng cấp cứu',
            'trạm cấp cứu',
            'phòng bác sĩ',
            'khu y tế',
            'điểm y tế',
            'phòng điều trị',
            'khu điều trị',
            'trạm khám bệnh',
            'nơi khám bệnh',
            'khu khám bệnh',
            'phòng khám bệnh',
            'điểm sơ cứu'],
 'dorm': ['ký túc xá',
          'kí túc xá',
          'ký túc',
          'kí túc',
          'khu nội trú',
          'nhà nội trú',
          'tòa nội trú',
          'dãy nội trú',
          'khối nội trú',
          'phòng nội trú',
          'khu lưu trú',
          'nhà lưu trú',
          'phòng lưu trú',
          'khu ký túc',
          'nhà ký túc',
          'tòa ký túc',
          'dãy ký túc',
          'khu KTX',
          'tòa KTX',
          'nhà KTX'],
 'gate': ['cổng trường',
          'cổng chính',
          'cổng phụ',
          'cổng vào',
          'cổng ra',
          'cổng ra vào',
          'cổng bảo vệ',
          'chốt cổng',
          'chốt gác',
          'trạm gác',
          'chốt bảo vệ',
          'trạm bảo vệ',
          'lối vào',
          'lối ra',
          'lối ra vào',
          'cửa vào',
          'cửa ra',
          'cửa ra vào',
          'cửa cổng',
          'cửa chính'],
 'lab': ['phòng thí nghiệm',
         'phòng thực nghiệm',
         'phòng thực hành',
         'phòng nghiên cứu',
         'phòng lab',
         'khu thí nghiệm',
         'khu thực nghiệm',
         'khu thực hành',
         'khu nghiên cứu',
         'xưởng thực hành',
         'xưởng thực nghiệm',
         'xưởng nghiên cứu',
         'phòng chế tạo',
         'xưởng chế tạo',
         'xưởng cơ khí',
         'phòng hóa nghiệm',
         'phòng kiểm nghiệm',
         'khu kiểm nghiệm',
         'trạm nghiên cứu',
         'viện nghiên cứu'],
 'lecture': ['giảng đường',
             'giảng phòng',
             'phòng học',
             'lớp học',
             'nhà học',
             'tòa học',
             'khu giảng đường',
             'dãy giảng đường',
             'tòa giảng đường',
             'phòng giảng',
             'phòng giảng dạy',
             'khu lớp học',
             'dãy lớp học',
             'dãy phòng học',
             'khu nhà học',
             'tòa nhà học',
             'phòng lý thuyết',
             'lớp lý thuyết',
             'phòng học chung',
             'phòng chuyên đề'],
 'library': ['thư viện',
             'thư khố',
             'kho sách',
             'phòng đọc',
             'phòng đọc sách',
             'khu đọc sách',
             'quầy mượn sách',
             'quầy trả sách',
             'phòng mượn sách',
             'phòng tham khảo',
             'kho tài liệu',
             'phòng tài liệu',
             'kho học liệu',
             'khu học liệu',
             'quầy thư viện',
             'phòng thư viện',
             'nhà thư viện',
             'khu thư viện',
             'tòa thư viện',
             'phòng tra cứu'],
 'office': ['văn phòng',
            'tòa hiệu bộ',
            'nhà hiệu bộ',
            'khu hiệu bộ',
            'phòng hành chính',
            'khu hành chính',
            'tòa hành chính',
            'phòng giáo vụ',
            'phòng đào tạo',
            'văn phòng khoa',
            'văn phòng trường',
            'phòng học vụ',
            'ban giám hiệu',
            'phòng giám hiệu',
            'phòng điều hành',
            'phòng quản lý',
            'phòng tiếp nhận',
            'quầy hành chính',
            'phòng công vụ',
            'ban quản lý'],
 'parking': ['bãi xe',
             'nhà xe',
             'hầm xe',
             'bãi đỗ xe',
             'bãi đậu xe',
             'bãi gửi xe',
             'bãi giữ xe',
             'nhà để xe',
             'nhà giữ xe',
             'hầm để xe',
             'hầm gửi xe',
             'hầm đỗ xe',
             'khu để xe',
             'khu gửi xe',
             'điểm gửi xe',
             'chỗ đỗ xe',
             'chỗ để xe',
             'chỗ gửi xe',
             'bãi xe máy',
             'bãi ô tô'],
 'sports': ['nhà thi đấu',
            'nhà thể chất',
            'khu thể thao',
            'sân thể thao',
            'sân bóng',
            'sân tập',
            'sân thi đấu',
            'khu thi đấu',
            'sân bóng đá',
            'sân bóng rổ',
            'sân cầu lông',
            'sân quần vợt',
            'phòng gym',
            'phòng tập',
            'khu tập luyện',
            'khu thể chất',
            'nhà luyện tập',
            'nhà tập luyện',
            'sân vận động',
            'sân tennis'],
 'near': ['gần nhất',
          'liền kề',
          'ngay cạnh',
          'sát cạnh',
          'kề bên',
          'sát bên',
          'ngay sát',
          'bên cạnh',
          'cạnh bên',
          'gần kề',
          'kề cận',
          'sát vách',
          'ở gần',
          'ở sát',
          'ở cạnh',
          'kề cạnh',
          'gần sát',
          'tiếp giáp',
          'lân cận',
          'ngay kế'],
 'far': ['xa nhất',
         'cách xa',
         'ở xa',
         'khá xa',
         'rất xa',
         'cực xa',
         'xa tít',
         'xa lắc',
         'xa xôi',
         'xa hẳn',
         'xa vời',
         'xa cách',
         'xa lơ',
         'xa ngái',
         'tít tắp',
         'tít mù',
         'cách biệt',
         'cách khá xa',
         'xa thật xa',
         'xa tít tắp'],
 'north': ['phía bắc',
           'hướng bắc',
           'mạn bắc',
           'mé bắc',
           'bên bắc',
           'khu bắc',
           'vùng bắc',
           'phía trên',
           'bên trên',
           'hướng trên',
           'mé trên',
           'đầu trên',
           'nửa trên',
           'theo hướng bắc',
           'ở phía bắc',
           'nằm phía bắc',
           'nằm bên trên',
           'thuộc phía bắc',
           'theo phía bắc',
           'tại mạn bắc'],
 'south': ['phía nam',
           'hướng nam',
           'mạn nam',
           'mé nam',
           'bên nam',
           'khu nam',
           'vùng nam',
           'phía dưới',
           'bên dưới',
           'hướng dưới',
           'mé dưới',
           'đầu dưới',
           'nửa dưới',
           'theo hướng nam',
           'ở phía nam',
           'nằm phía nam',
           'nằm bên dưới',
           'thuộc phía nam',
           'theo phía nam',
           'tại mạn nam'],
 'east': ['phía đông',
          'hướng đông',
          'mạn đông',
          'mé đông',
          'bên đông',
          'khu đông',
          'vùng đông',
          'phía phải',
          'bên phải',
          'hướng phải',
          'mé phải',
          'cạnh phải',
          'nửa phải',
          'theo hướng đông',
          'ở phía đông',
          'nằm phía đông',
          'nằm bên phải',
          'thuộc phía đông',
          'theo phía đông',
          'tại mạn đông'],
 'west': ['phía tây',
          'hướng tây',
          'mạn tây',
          'mé tây',
          'bên tây',
          'khu tây',
          'vùng tây',
          'phía trái',
          'bên trái',
          'hướng trái',
          'mé trái',
          'cạnh trái',
          'nửa trái',
          'theo hướng tây',
          'ở phía tây',
          'nằm phía tây',
          'nằm bên trái',
          'thuộc phía tây',
          'theo phía tây',
          'tại mạn tây'],
 'urgent': ['khẩn cấp',
            'cấp bách',
            'khẩn thiết',
            'khẩn trương',
            'hỏa tốc',
            'tốc hành',
            'cấp tốc',
            'tức tốc',
            'gấp rút',
            'gấp gáp',
            'cần gấp',
            'giao gấp',
            'giao ngay',
            'làm ngay',
            'gửi ngay',
            'chuyển gấp',
            'sát giờ',
            'phải gấp',
            'sắp muộn',
            'ưu tiên cao'],
 'fragile': ['dễ vỡ',
             'dễ bể',
             'dễ nứt',
             'dễ gãy',
             'dễ hỏng',
             'dễ sứt',
             'dễ móp',
             'dễ trầy',
             'dễ xước',
             'mong manh',
             'thủy tinh',
             'đồ gốm',
             'đồ sứ',
             'đồ kính',
             'hàng kính',
             'hàng sứ',
             'đồ dễ vỡ',
             'hàng dễ vỡ',
             'tránh va đập',
             'nhẹ tay'],
 'none': ['không có',
          'không rõ',
          'chưa rõ',
          'không biết',
          'chưa biết',
          'chưa có',
          'bỏ qua',
          'không cần',
          'để trống',
          'chưa xác định',
          'không xác định',
          'chưa thấy',
          'không thấy',
          'không tồn tại',
          'không tìm thấy',
          'không cung cấp',
          'chưa cung cấp',
          'chưa ghi',
          'chưa đề cập',
          'không đề cập']}



SESSION = requests.Session()

SESSION.headers.update({

    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/154.0 Safari/537.36',

    'Accept-Language': 'vi,en;q=0.8',

})



def normalize_wiki_word(text):

    if text is None: return ''

    text = str(text).lower().strip()

    text = re.sub(r'\s*\([^)]*\)', '', text)

    text = re.sub(r'^[*•\-–—]+\s*', '', text)

    text = re.sub(r'\s+', ' ', text)

    text = re.sub(r'\s+([,.!?;:])', r'\1', text)

    return text.strip()



if 'fold' not in globals():

    import unicodedata

    def fold(text):

        text = str(text).lower().strip()

        text = unicodedata.normalize('NFD', text)

        text = ''.join(c for c in text if unicodedata.category(c) != 'Mn')

        text = text.replace('đ', 'd').replace('đ', 'd')

        return re.sub(r'\s+', ' ', text).strip()



GLOBAL_BAD_EXACT = {'trên', 'dưới', 'trái', 'phải', 'trong', 'ngoài'}

CLASS_BAD_WORDS = {

    'canteen': {'ăn', 'uống', 'thực phẩm', 'thức ăn', 'đồ ăn', 'cơm'},

    'clinic': {'bệnh', 'đau', 'chữa', 'thuốc', 'y'},

    'dorm': {'nhà', 'ở', 'ngủ', 'phòng'},

    'gate': {'cửa', 'cổng', 'lối'},

    'lab': {'nghiên cứu', 'thí nghiệm'},

    'lecture': {'học', 'lớp', 'dạy'},

    'library': {'sách', 'đọc'},

    'office': {'làm', 'việc', 'phòng'},

    'parking': {'xe', 'đỗ', 'đậu'},

    'sports': {'thể', 'thao', 'vận', 'động', 'chơi'},

    'near': {'xa'}, 'far': {'gần'},

    'north': {'nam', 'đông', 'tây'}, 'south': {'bắc', 'đông', 'tây'},

    'east': {'bắc', 'nam', 'tây'}, 'west': {'bắc', 'nam', 'đông'}

}



def valid_synonym(word, cls):

    word = normalize_wiki_word(word)

    if not word or len(word) < 2 or len(word) > 60: return False

    if re.search(r'\d', word): return False

    if not re.search(r'[a-zA-ZÀ-ỹĐđ]', word): return False

    if word in GLOBAL_BAD_EXACT: return False

    if word in CLASS_BAD_WORDS.get(cls, set()): return False

    return True



def scrape_wiktionary(word):

    word = normalize_wiki_word(word)

    if not word: return []

    url = 'https://vi.wiktionary.org/wiki/' + urllib.parse.quote(word.replace(' ', '_'))

    try:

        response = SESSION.get(url, timeout=REQUEST_TIMEOUT)

        if response.status_code != 200: return []

        soup = BeautifulSoup(response.text, 'html.parser')

        results = []

        target_keywords = ['từ đồng nghĩa', 'đồng nghĩa', 'từ liên quan', 'liên quan', 'xem thêm']

        for tag in soup.find_all(['h2', 'h3', 'h4', 'span']):

            txt = tag.get_text(' ', strip=True).lower()

            if any(keyword in txt for keyword in target_keywords):

                parent = tag.find_parent() if tag.name == 'span' else tag

                if parent:

                    curr = parent.next_sibling

                    while curr:

                        if curr.name in ['h2', 'h3', 'h4']: break

                        if curr.name == 'ul':

                            for li in curr.find_all('li', recursive=False):

                                text_li = re.sub(r'\([^)]*\)', '', li.get_text(' ', strip=True))

                                for part in re.split(r'[;,/]', text_li):

                                    p = normalize_wiki_word(part)

                                    if p: results.append(p)

                        curr = curr.next_sibling

        return list(dict.fromkeys(results))

    except Exception:

        return []



def crawl_class(cls, seeds, max_depth=MAX_DEPTH, max_words=MAX_WORDS_PER_CLASS):

    collected, queue, seen = set(), [], set()

    for seed in seeds:

        seed = normalize_wiki_word(seed)

        if seed:

            collected.add(seed)

            queue.append((seed, 0))

    while queue:

        word, depth = queue.pop(0)

        key = fold(word)

        if key in seen: continue

        seen.add(key)

        if depth >= max_depth: continue

        if len(collected) >= max_words: break

        synonyms = scrape_wiktionary(word)

        time.sleep(WIKI_SLEEP)

        for synonym in synonyms:

            synonym = normalize_wiki_word(synonym)

            if not valid_synonym(synonym, cls) or synonym in collected: continue

            collected.add(synonym)

            queue.append((synonym, depth + 1))

            if len(collected) >= max_words: break

    return collected



print('\n' + '=' * 75 + '\nBẮT ĐẦU CÀO WIKTIONARY\n' + '=' * 75)

WIKI_RAW = {}

for cls in WIKI_SEEDS:

    words = crawl_class(cls, WIKI_SEEDS[cls])

    WIKI_RAW[cls] = words

    print(f'[{cls}] => {len(words)} alias')



SPECIAL_ALIASES = {} # Fix lỗi undefine của user

for cls, aliases in SPECIAL_ALIASES.items():

    if cls not in WIKI_RAW: WIKI_RAW[cls] = set()

    for alias in aliases:

        a = normalize_wiki_word(alias)

        if a: WIKI_RAW[cls].add(a)



WIKI_CLEAN = {}

for cls, words in WIKI_RAW.items():

    clean = set()

    for word in words:

        word = normalize_wiki_word(word)

        if not word: continue

        if cls in {'yes', 'no'} and len(word) <= 40: clean.add(word)

        elif valid_synonym(word, cls): clean.add(word)

    WIKI_CLEAN[cls] = clean



for cls, seeds in WIKI_SEEDS.items():

    for seed in seeds:

        s = normalize_wiki_word(seed)

        if s: WIKI_CLEAN.setdefault(cls, set()).add(s)



if 'BANK' not in globals(): BANK = {}

for cls in WIKI_CLEAN.keys(): BANK.setdefault(cls, [])



existing_owner = {}

for existing_cls, aliases in BANK.items():

    for alias in aliases:

        fw = fold(alias)

        if fw: existing_owner.setdefault(fw, existing_cls)



for cls, words in WIKI_CLEAN.items():

    words = sorted(words, key=lambda x: (-len(x), x))

    for word in words:

        fw = fold(word)

        if not fw: continue

        if fw in existing_owner and existing_owner[fw] != cls: continue

        if word not in BANK[cls]:

            BANK[cls].append(word)

            existing_owner[fw] = cls



for cls in BANK: BANK[cls] = sorted(set(BANK[cls]), key=lambda x: (-len(x), fold(x)))

print("Đã hoàn tất cập nhật BANK từ Wiktionary!")


BẮT ĐẦU CÀO WIKTIONARY
[canteen] => 20 alias
[clinic] => 20 alias
[dorm] => 20 alias
[gate] => 20 alias
[lab] => 20 alias
[lecture] => 20 alias
[library] => 20 alias
[office] => 20 alias
[parking] => 20 alias
[sports] => 20 alias
[near] => 20 alias
[far] => 20 alias
[north] => 20 alias
[south] => 20 alias
[east] => 20 alias
[west] => 20 alias
[urgent] => 20 alias
[fragile] => 20 alias
[none] => 20 alias
Đã hoàn tất cập nhật BANK từ Wiktionary!


In [ ]:
# ADD-ON NGOÀI CELL WIKI GỐC: lọc alias mơ hồ SAU KHI toàn bộ crawl gốc và print đã kết thúc.
# Không chạm tới MAX_DEPTH/MAX_WORDS_PER_CLASS/sleep/crawl/print của code bạn gửi.
# VD: Wiki truy hồi “ăn chơi”, “ăn học” -> KHÔNG đưa vào BANK['canteen'].
import collections, re
ALIAS_ANCHOR = {
 'canteen': ['căng tin','căn tin','nhà ăn','quán ăn','quán cơm','tiệm cơm','tiệm ăn','quầy cơm','quầy đồ ăn','khu ẩm thực','bếp ăn','phòng ăn','suất ăn','quầy ăn uống','giải khát','nhà bếp','quầy thức ăn'],
 'clinic': ['phòng khám','trạm y tế','phòng y tế','trạm xá','bệnh xá','sơ cứu','cấp cứu','phòng bác sĩ','điều trị','khám bệnh','khám chữa','y tế'],
 'dorm': ['ký túc','kí túc','nội trú','lưu trú','khu ktx','tòa ktx','nhà ktx'],
 'gate': ['cổng trường','cổng chính','cổng phụ','cổng vào','cổng ra','cổng bảo vệ','chốt gác','trạm gác','bảo vệ','lối ra vào','cửa ra vào'],
 'lab': ['thí nghiệm','thực nghiệm','phòng lab','thực hành','nghiên cứu','chế tạo','hóa nghiệm','kiểm nghiệm'],
 'lecture': ['giảng đường','giảng phòng','phòng học','lớp học','nhà học','tòa học','giảng dạy','lý thuyết','chuyên đề'],
 'library': ['thư viện','thư khố','kho sách','phòng đọc','mượn sách','trả sách','học liệu','tra cứu'],
 'office': ['văn phòng','hiệu bộ','hành chính','giáo vụ','đào tạo','học vụ','giám hiệu','điều hành','phòng quản lý','quầy hành chính'],
 'parking': ['bãi xe','nhà xe','hầm xe','đỗ xe','đậu xe','gửi xe','giữ xe','để xe','xe máy','ô tô'],
 'sports': ['thể thao','thể chất','thi đấu','sân bóng','sân vận động','cầu lông','quần vợt','phòng gym','phòng tập','tập luyện','luyện tập','sân tennis'],
}
# Important: giữ toàn bộ 20 seed, dù có từ viết tắt/nhiều cách diễn đạt.
# Các từ thu thập khác phải có cụm đặc trưng, không chứa từ quá chung.
def _vocab_norm(s):
    return re.sub(r'\s+', ' ', fold(str(s)).lower()).strip()

BANK_BEFORE_AUDIT={k:list(v) for k,v in BANK.items()}
removed=collections.Counter()
for cls,terms in ALIAS_ANCHOR.items():
    required={_vocab_norm(v) for v in WIKI_SEEDS[cls]}
    anchors=[_vocab_norm(v) for v in terms]
    keep=[]
    for term in BANK[cls]:
        w=_vocab_norm(term)
        # all user-curated seeds and known explicit synonyms are retained; newly crawled expressions
        # should have a full contextual root (not single broad word like 'ăn' or 'học').
        allowed=(w in required or (2 <= len(w.split()) <= 4 and any(a in w for a in anchors)))
        if allowed:keep.append(term)
        else:removed[cls]+=1
    BANK[cls]=list(dict.fromkeys(keep))
# Prevent false antonyms from auto-crawl, but keep original 20 seeds exactly.
for cls in ('near','far','north','south','east','west','urgent','fragile','none'):
    seed_set={_vocab_norm(w) for w in WIKI_SEEDS[cls]}
    # Relation/attribute words returned by a 10-depth search could be antonyms,
    # so an external review is required before promoting them to training alias.
    extras={_vocab_norm(w) for w in BANK.get(cls,[]) if _vocab_norm(w) not in seed_set}
    removed[cls]+=len(extras)
    BANK[cls]=list(dict.fromkeys(WIKI_SEEDS[cls]))
if '_ALIAS_CACHE' in globals():_ALIAS_CACHE.clear()
if '_CLASH' in globals():_CLASH.clear()
print('AUDIT LỌC NGHĨA (sau crawl Wiki gốc):',dict(removed))
print('BANK SAU AUDIT:',{cls:len(BANK[cls]) for cls in WIKI_SEEDS})
assert all(all(w in BANK[cls] for w in WIKI_SEEDS[cls]) for cls in WIKI_SEEDS)
assert 'ăn chơi' not in BANK['canteen'] and 'ăn học' not in BANK['canteen']


AUDIT LỌC NGHĨA (sau crawl Wiki gốc): {'dorm': 1, 'gate': 1, 'near': 0, 'far': 0, 'north': 0, 'south': 0, 'east': 0, 'west': 0, 'urgent': 0, 'fragile': 0, 'none': 0}
BANK SAU AUDIT: {'canteen': 20, 'clinic': 20, 'dorm': 23, 'gate': 20, 'lab': 20, 'lecture': 20, 'library': 21, 'office': 21, 'parking': 20, 'sports': 21, 'near': 20, 'far': 20, 'north': 20, 'south': 20, 'east': 20, 'west': 20, 'urgent': 20, 'fragile': 20, 'none': 20}


In [ ]:
# ================================================================

# GENERATOR V3 - GHÉP CÂU THEO THÀNH PHẦN (COMPOSITIONAL)

# ================================================================

# Giả định các biến sau đã tồn tại trong notebook (giống V2):

#   BANK, PLACES, KINDS, ITEMS, H2I, base_items, fold

#

# Output giữ nguyên schema:  {'text': ..., 'y': ..., 'syn': True}

# Metadata (loại câu, vị trí goal/distractor...) nằm ở GEN_META[text]

# để phân tích lỗi, KHÔNG nhét vào item nên không phá collate cũ.

#

# Ý tưởng chính:

#   1. Không còn "1 mẫu = 1 câu". Mỗi câu = persona + khung cú pháp +

#      động từ/giới từ + tân ngữ + particle + (câu lừa) + (via) + (attr)

#      + (câu đệm) -> hàng chục nghìn skeleton khác nhau.

#   2. Địa điểm được che bằng placeholder @0\@..\@6@ trong lúc ghép,

#      chỉ thay bằng alias thật ở cuối -> nhiễu/viết tắt/typo không

#      bao giờ làm hỏng entity, và label luôn khớp với text.

#   3. Câu lừa/phủ định là các KIỂU NGỮ NGHĨA (tự sửa, thông tin cũ,

#      đơn khác, nơi đóng cửa, ghi chú sai, hỏi-đáp...), không phải

#      chỉ là "đừng giao ở X".

#   4. Có thể giữ riêng một số kiểu câu làm tập test OOD

#      (goal_types / exclude_goal_types) để đo model hiểu hay học thuộc.

# ================================================================



import random

import re

import unicodedata

import collections





# ---------------------------------------------------------------

# 0. CẤU HÌNH (chỉnh tỉ lệ ở đây)

# ---------------------------------------------------------------

CFG = {

    "p_kind": 0.48,            # địa điểm có kind/anchor

    "p_via": 0.65,             # câu có điểm lấy hàng (via)

    "p_via_composite": 0.50,   # via nằm chung 1 câu với goal

    "p_trap_rel": 0.20,        # câu lừa mô tả địa điểm kèm kind/anchor

    "p_extra_aff": 0.30,       # thêm câu "đi ngang qua" (không dừng)

    "p_extra_neg": 0.50,       # thêm câu phủ định phụ

    "p_neg_goal": 0.80,        # bucket Neg: goal-unit tự mang phủ định

    "p_filler_pre": 0.18,

    "p_filler_post": 0.22,

    "p_filler_info": 0.20,

    "p_attr_combined": 0.30,   # urgent+fragile gộp 1 câu

    "p_omit_no": 0.12,         # label 'no' nhưng không nhắc tới (thực tế hay vậy)

    "p_attr_hard": 0.22,       # câu attr khó (nhắc tới nhưng của đơn khác...)

    "p_attr_para": 0.15,       # fragile nói bằng từ KHÔNG nằm trong BANK['fragile']

    "use_para_attr": True,     # tắt nếu guideline gán nhãn fragile của bạn khác

    "p_chat": 0.24,            # văn phong chat: viết thường, ko/đc/vs...

    "p_terse": 0.10,           # văn phong ghi chú rời rạc "a | b | c"

    "p_lower": 0.12,

    "p_sentence_typo": 0.60,   # (bucket Typo) typo ở phần ngữ cảnh

    "p_alias_typo": 0.22,      # (bucket Typo) typo ở tên địa điểm

}



PH = {"G": "@0@", "V": "@1@", "TV": "@2@",

      "T1": "@3@", "T2": "@4@", "T3": "@5@", "T4": "@6@"}

PH_IDX = {"G": 0, "V": 1, "TV": 2, "T1": 3, "T2": 4, "T3": 5, "T4": 6}





# ---------------------------------------------------------------

# 1. NGÂN HÀNG TỪ VỰNG / PERSONA

# ---------------------------------------------------------------

# (me, you, miền_nam, lịch_sự)

PERSONAS = [

    ("mình", "bạn", False, False), ("mình", "bạn", True, False),

    ("tớ", "cậu", False, False), ("tui", "anh", True, False),

    ("tui", "chị", True, False), ("em", "anh", False, True),

    ("em", "chị", False, True), ("em", "bác", True, True),

    ("tôi", "anh", False, True), ("tôi", "chị", False, True),

    ("anh", "em", False, False), ("chị", "em", False, False),

    ("bên mình", "bạn", False, False), ("shop", "anh shipper", False, True),

    ("shop", "bạn shipper", True, False), ("bên em", "anh tài xế", False, True),

]



NAMES = ["chị Lan", "anh Nam", "bạn Minh", "cô Hoa", "chú Tùng", "em Linh",

         "anh Hùng", "chị Mai", "bạn Khánh", "bác Dũng", "anh Phúc", "chị Thảo"]



TAIL_COMMON = ["với", "giúp {me}", "giúp {me} với", "nhớ nhé"]

TAIL_NORTH = ["nhé", "nhá"]

TAIL_SOUTH = ["nha", "nghen", "hen", "dùm {me}", "giùm {me}", "nhen"]

TAIL_POLITE = ["ạ", "nhé ạ", "với ạ"]

TAIL_END_ONLY = ["được không", "được chứ"]



OBJ_PRON = ["đơn này", "món này", "hàng này", "kiện hàng này", "gói này",

            "đơn hàng này", "đồ này", "nó"]



# động từ giao -> các giới từ đi kèm hợp ngữ pháp

GIVE_VERBS = {

    "giao": ["tới", "đến", "ở", "tại", "về", "sang"],

    "chuyển": ["tới", "đến", "về", "sang"],

    "đưa": ["tới", "đến", "về", "sang"],

    "mang": ["tới", "đến", "về", "sang"],

    "gửi": ["tới", "đến", "về", "sang"],

    "vận chuyển": ["tới", "đến", "về"],

    "ship": ["tới", "đến", "về"],

    "chở": ["tới", "đến", "về"],

    "đem": ["tới", "đến", "về", "sang"],

    "để lại": ["ở", "tại"],

    "bàn giao": ["tại", "ở", "tới"],

}



GS_FAMS = ["imp", "nho", "you", "want", "pass", "topic", "formal",

           "nominal", "recv", "frag", "go"]



NOMINAL = ["điểm giao: {G}", "điểm giao là {G}", "địa chỉ giao: {G}",

           "địa chỉ giao hàng là {G}", "điểm đến: {G}",

           "điểm đến cuối cùng là {G}", "nơi giao hàng: {G}",

           "chỗ giao là {G}", "đích đến của đơn này là {G}",

           "địa chỉ người nhận: {G}", "giao hàng tại: {G}"]

RECV = ["người nhận đang đợi sẵn ở {G}", "khách đang chờ ở {G}",

        "người nhận ở {G}", "khách ở {G} nhé",

        "người nhận đang đứng chờ tại {G}", "bên nhận trực tại {G}",

        "có người chờ nhận hàng ở {G}"]

FRAG = ["{G} {t}", "tới {G} {t}", "→ {G}", "ship {G} {t}", "giao {G}",

        "đi {G} {t}", "{G} nhé", "đến {G} {t}"]

GO = ["đơn này đi {G}", "chuyến này chạy tới {G}",

      "chuyến này đích đến là {G}", "đơn đi {G} {t}",

      "chạy {G} giúp {me}", "đích của đơn này là {G}"]



# lấy hàng (via)

PICK_PATTERNS = [

    "lấy {o} ở {V}", "lấy {o} tại {V}", "lấy {o} từ {V}", "ghé {V} lấy {o}",

    "qua {V} lấy {o}", "tạt qua {V} lấy {o}", "đến {V} lấy {o}",

    "chạy qua {V} lấy {o}", "nhận {o} tại {V}", "nhận {o} từ {V}",

    "ghé {V} nhận {o}", "gom {o} ở {V}", "bốc {o} ở {V}",

    "dừng ở {V} lấy {o}", "pick {o} ở {V}", "qua {V} bốc {o}",

    "ghé {V} gom {o}",

]

STOP_PATTERNS = ["ghé {V} một lát", "dừng ở {V} một chút", "tạt qua {V} trước",

                 "có một điểm dừng ở {V}", "dừng chân ở {V} trước đã"]

PICK_OBJ = ["hàng", "đồ", "đơn", "đơn hàng", "kiện hàng", "{item}",

            "món {item}", "đơn {item}", "món này", "đơn này", ""]

PS_FRAMES = [

    "{PV} {tail}", "nhờ {you} {PV} {tail}", "{You} {PV} {tail}",

    "trước tiên {PV}", "đầu tiên {PV} {tail}", "{me} cần {you} {PV}",

    "điểm lấy hàng: {V}", "nơi lấy hàng là {V}",

    "điểm dừng đầu tiên: {V}", "{o} đang để ở {V}", "hàng đang nằm ở {V}",

    "{o} gửi sẵn ở {V}", "phiền {you} {PV} {tail}", "vui lòng {PV}",

    "{o} hiện ở {V}, qua đó lấy giúp {me}",

]



CORR_SOFT = ["à nhầm", "nhầm rồi", "ủa lộn", "lộn rồi", "xin lỗi, {me} nói nhầm",

             "khoan, sửa lại", "đính chính một chút", "{me} sửa lại",

             "ý lộn", "viết nhầm", "gõ nhầm", "oops", "sorry nhầm",

             "để {me} nói lại", "nói lại cho đúng", "khoan đã"]

CORR_NEG = ["à không", "bỏ câu trước nhé", "quên chỗ vừa nói đi",

            "đừng nghe câu trước", "khỏi làm theo câu trên"]



REDIR_FORMS = ["{G} mới đúng", "là {G} cơ", "tới {G} nhé", "giao {G} mới phải",

               "ý {me} là {G}", "{G} mới là chỗ nhận", "phải là {G}",

               "chuyển sang {G}", "đổi thành {G}", "đổi qua {G}",

               "sửa thành {G}", "{GV}", "{GV}", "thật ra là {G}",

               "đúng ra là {G}"]



WHY = ["đang đóng cửa", "đã hết giờ làm việc", "đang sửa chữa",

       "đang bị rào chắn", "đang ngập nước", "đang bảo trì", "vừa khóa cổng",

       "đang phong tỏa", "vừa bị cắt điện", "đang vắng người",

       "đã đóng cổng từ chiều"]

CONV = ["tiện đường hơn", "dễ đi hơn", "đỡ kẹt xe hơn", "rẻ phí ship hơn",

        "có chỗ đậu xe hơn", "vào dễ hơn"]



NOT_T = ["chứ không phải {T}", "không phải {T} nhé", "đừng nhầm sang {T}",

         "{T} thì thôi", "khỏi qua {T}", "{T} là sai chỗ", "đừng đưa tới {T}",

         "{T} không liên quan", "tránh {T} nhé", "{T} thì bỏ qua"]

NOT_LEAD = ["không phải {T}", "đừng giao ở {T}", "bỏ {T} đi", "{T} thì không",

            "khỏi qua {T}", "đừng để hàng ở {T}", "tuyệt đối không giao ở {T}",

            "đừng mang tới {T}", "{T} sai rồi", "tránh {T} giúp {me}",

            "{T} không phải chỗ nhận", "không giao ở {T} nhé",

            "đừng nhầm với {T}"]



QA_Q = ["giao ở {T} đúng không?", "tới {T} phải không?", "chỗ nhận là {T} hả?",

        "{T} là điểm giao đúng không?", "giao {T} nhỉ?", "là {T} đúng không ạ?",

        "{T} nhé?", "đơn này đi {T} hả?"]

QA_A = ["không, {GS}", "không phải, {GS}", "sai rồi, {GS}", "không nha, {GS}",

        "không đâu, {GS}", "nhầm rồi, {GS}", "không, {G} cơ",

        "không phải {T}, là {G}", "chưa đúng, {GS}", "ủa không, {GS}",

        "{T} là sai, {GS}"]

QA_REV = ["{GS}. {T} hả? Không phải nhé", "{GS} — còn {T} á? không phải nha",

          "{GS}, {T} thì không nha", "{GS}. {T} á? Không đúng đâu"]



COMPOSITE = [

    "{PV}, rồi {GV}", "{PV}, sau đó {GV}", "sau khi {PV} xong thì {GV}",

    "{PV}; xong {GV}", "{GS}, nhưng trước đó {PV}", "trước khi {GV} nhớ {PV}",

    "việc đầu tiên: {PV}. Việc cuối: {GV}", "{PV} trước rồi mới {GV}",

    "từ {V}, {GV}", "{G} là nơi nhận cuối, còn {V} là chỗ lấy hàng",

    "chặng 1: {PV}. Chặng 2: {GV}", "{GS} (nhớ {PV} trước)",

    "{PV}, rồi {GV} {tail}",

]



GOAL_TPL = {

    "self_correct": [

        "{GV_T}, {corr}, {REDIR}", "{GV_T}... {corr}, {REDIR}",

        "{GS_T}. {Corr}, {REDIR}", "{T}... {corr}, {REDIR}",

        "{T} — {corr}, {REDIR}", "lúc nãy {me} nói {T} là nói nhầm, {REDIR}",

        "lúc đầu {me} bảo {T}, {corr}, {REDIR}",

        "{Corr} nhé, {T} là {me} gõ nhầm, {REDIR}",

        "{GS}. {T} ở tin trước là {me} nhập nhầm",

        "{GS}. Còn {T} lúc nãy là nhầm, quên nó đi",

        "{T} nhầm, {REDIR}", "{T}, ý lộn, {REDIR}",

        "gõ nhầm địa chỉ rồi: {T} sai, {REDIR}",

        "{G} mới đúng nha, lúc nãy ghi {T} là gõ nhầm",

        "{GS}. Vừa rồi {me} ghi {T} là nhầm",

        "{GV}, còn {T} là nhập nhầm ở tin trước",

    ],

    "old_info": [

        "lần trước {GV_T}, lần này {REDIR}",

        "địa chỉ cũ là {T}, địa chỉ mới là {G}",

        "khách vừa chuyển từ {T} sang {G}, nên {GV}",

        "hôm qua {GV_T}; hôm nay {GV}",

        "đơn hôm trước ở {T}, còn hôm nay {GV}",

        "{T} là chỗ cũ rồi, giờ thì {REDIR}",

        "ghi chú cũ ghi {T}, thông tin mới nhất là {G}",

        "trước đây khách nhận ở {T}, hiện tại {REDIR}",

        "lúc đặt hàng khách chọn {T}, nhưng sau đó đổi sang {G}",

        "sáng nay {me} định gửi tới {T}, giờ thì {REDIR}",

        "{GS}. {T} chỉ là địa chỉ cũ",

        "từ giờ {GV}; {T} là chuyện của tuần trước",

        "đổi địa chỉ rồi nhé: từ {T} sang {G}",

        "{G} là địa chỉ mới, {T} là địa chỉ cũ",

        "{GS} (khách đã dọn khỏi {T})",

        "{GV} nhé, {T} là chỗ cũ rồi",

        "giờ {GV}; địa chỉ cũ ({T}) bỏ đi",

    ],

    "unavailable": [

        "{T} {why}, nên {GV}", "vì {T} {why} nên {GV}",

        "{T} {why} rồi, {REDIR}", "{GS}, vì {T} {why}",

        "chỗ {T} {why}, {me} đổi sang {G}", "{T} {why} — {GV}",

        "nghe nói {T} {why}, nên cứ {GV}",

        "{T} {why}; phương án thay thế là {G}",

        "{GS} vì {T} {why}",

        "{GV} nhé — {T} {why} rồi",

        "đổi sang {G} thay cho {T}, vì chỗ đó {why}",

    ],

    "preference_override": [

        "{T} {conv}, nhưng cứ {GV}", "dù {T} {conv} thì vẫn {GV}",

        "{T} {conv} đấy, nhưng người nhận chỉ ở {G}",

        "{T} {conv} thật, song điểm nhận là {G}", "{GS}, dù {T} {conv}",

        "tuy {T} {conv} nhưng đích đến vẫn là {G}",

        "mặc dù {T} {conv}, cứ {GV} giúp {me}",

    ],

    "other_order": [

        "đơn của {name} thì {GV_T}, còn đơn này {GV}",

        "{GS} (đơn của {name} thì ở {T})",

        "hai đơn đi hai nơi: đơn của {name} tới {T}, đơn này tới {G}",

        "đơn kia {GV_T}, đơn này {GV}",

        "trong cùng chuyến có đơn đi {T}, riêng đơn này {GV}",

        "{name} nhận ở {T}; riêng đơn này thì {GV}",

        "đơn này đi {G}; đơn của {name} đi {T}, hai đơn khác nhau",

        "{GV}; còn đơn của {name} thì đi {T}",

    ],

    "reject_goal": [

        "{GS}, {not_T}", "{GS}; {not_T}", "{notT_lead}. {GS}",

        "{notT_lead}, {GS}", "{GV_C}", "{GV_C} {tail}",

        "giữa {T} với {G}, {GV}",

        "có hai chỗ {T} và {G}; {G} mới là chỗ nhận, {T} bỏ qua",

        "đừng lẫn {T} với {G}: {GV}", "chỉ {G} thôi, {T} thì không",

        "{G} chứ không phải {T}, nhớ giúp {me}",

        "đừng đưa nhầm qua {T}, {GS}", "{T} là chỗ khác, {GS}",

    ],

    "note_wrong": [

        "ghi chú ghi {T} là ghi sai, {GS}",

        "app đang hiện {T} nhưng đó là địa chỉ lỗi, {GS}",

        "đừng làm theo địa chỉ trên hệ thống ({T}); {GS}",

        "địa chỉ trên đơn ({T}) không còn đúng, {GS}",

        "tin nhắn trước nhắc {T}, bỏ qua tin đó, {GS}",

        "người ta hay nhầm sang {T}, nhưng {GV}",

        "thông tin {T} lúc nãy là sai, {GS}",

        "khách ghi {T} nhưng thực tế không nhận ở đó; {GS}",

        "bản đồ đang chỉ {T}, đừng đi theo; {GS}",

        "đừng tin hệ thống, nó ghi {T} là sai. {GS}",

        "{GS}; địa chỉ {T} trên hệ thống là sai",

        "{GS}. Dòng ghi {T} trong ghi chú là lỗi, bỏ qua giúp {me}",

        "{GV}, đừng theo địa chỉ {T} trong ghi chú",

        "{GS}. Chỗ ghi {T} trên hệ thống là sai",

    ],

    "reject_multi": [

        "không giao ở {T1}, cũng không phải {T2}; {GS}",

        "{T1} với {T2} đều không phải điểm nhận, {GS}",

        "loại {T1} và {T2} khỏi danh sách; {GS}",

        "chỉ {G} thôi, {T1} hay {T2} đều bỏ qua",

        "cả {T1} lẫn {T2} đều sai, {GS}",

        "{T1}, {T2}, {T3} đều không phải; {GS}",

        "{GS} — không phải {T1}, không phải {T2}",

        "{GS}, còn {T1} hay {T2} thì khỏi",

        "trong mấy chỗ {T1}, {T2}, {G}, chỉ {G} là đúng",

    ],

}



AFF_W = {"plain": 3.0, "self_correct": 2.0, "old_info": 1.5, "unavailable": 1.5,

         "preference_override": 1.5, "other_order": 1.5}

NEG_W = {"reject_goal": 3.0, "qa_reject": 1.5, "note_wrong": 1.5,

         "reject_multi": 1.5}



EXTRA_TPL = {

    "route_pass": [

        "đường đi sẽ ngang {T}, cứ chạy thẳng", "chạy ngang {T} là thấy",

        "{T} chỉ là mốc để định hướng", "lấy {T} làm mốc rồi đi tiếp",

        "đi ngang qua {T} sẽ thấy", "{T} nằm trên đường đi thôi",

    ],

    "route_no_stop": [

        "đi ngang {T} thôi, đừng dừng", "chạy qua {T} nhưng không cần ghé",

        "{T} chỉ để định hướng, không dừng ở đó",

        "không cần ghé {T}, chỉ đi ngang", "{T} chỉ là điểm đi ngang, không dừng",

        "lướt qua {T} thôi nhé, không vào",

    ],

    "no_stop": [

        "không cần ghé đâu hết, đi thẳng", "đi thẳng, đừng ghé {T}",

        "khỏi qua {T} lấy gì, hàng có sẵn trên xe rồi",

        "hàng có sẵn rồi, không cần qua {T} lấy",

        "không cần lấy hàng ở {T}, đi thẳng",

        "không có điểm dừng trung gian nào, kể cả {T}",

        "đi một mạch, không ghé {T}",

        "không phải qua {T} đâu, hàng {you} đã cầm rồi",

    ],

    "reject_via": [

        "lấy hàng ở {V}, đừng lấy ở {TV}", "{PS}, không phải {TV}",

        "nhận hàng tại {V} chứ không phải {TV}", "{TV} khỏi ghé, {PS}",

        "không lấy hàng ở {TV} nữa, {PS}",

        "điểm lấy hàng là {V}; {TV} là chỗ cũ, bỏ qua",

        "đừng ghé {TV}; {PV}", "{TV} thì thôi, qua {V} lấy hàng",

        "hàng không nằm ở {TV} mà ở {V}",

    ],

}



# ---- thuộc tính urgent / fragile ------------------------------

ATTR = {

    ("urgent", "yes", "std"): [

        "đơn này gấp", "gấp lắm", "gấp nhé", "đơn gấp", "cần giao gấp",

        "{me} đang cần gấp", "làm nhanh giúp {me}", "ưu tiên đơn này trước",

        "đơn này ưu tiên số một", "khách đang đứng chờ rồi",

        "khách chờ lâu rồi", "cần có trong {n} phút nữa",

        "trong vòng {n} phút phải tới", "càng sớm càng tốt",

        "giao ASAP nha", "đơn hỏa tốc", "đơn khẩn", "sát giờ lắm rồi",

        "đi liền giúp {me}", "chạy nhanh giúp {me}",

        "trễ là khách hủy đơn đó",

    ],

    ("urgent", "no", "std"): [

        "đơn này không gấp", "không gấp đâu", "chưa cần gấp", "không cần vội",

        "từ từ cũng được", "cứ thong thả", "khi nào tiện thì giao",

        "trong ngày mai cũng được", "giao chiều mai cũng ổn",

        "cuối tuần giao cũng không sao", "không có yêu cầu ưu tiên",

        "đi theo lịch bình thường là được", "đơn thường, không phải đơn gấp",

        "khỏi chạy nhanh", "không cần giao ngay", "{me} không vội",

        "thời gian thoải mái", "không áp lực thời gian",

        "ngày mốt tới cũng được", "tuần này giao xong là được",

    ],

    ("urgent", "yes", "hard"): [

        "không thể để trễ được", "không được chậm nhé",

        "tuyệt đối không được trễ", "đừng để khách chờ thêm",

        "không còn nhiều thời gian đâu", "chớ để trễ nhé",

        "nhỏ thôi nhưng cần gấp", "đồ nhỏ mà gấp lắm",

        "tuy đơn nhỏ nhưng khách chờ rồi",

    ],

    ("urgent", "no", "hard"): [

        "đơn của {name} mới gấp, đơn này thì không",

        "hôm qua thì gấp, hôm nay đã ổn rồi", "đừng tưởng đây là đơn gấp",

        "{name} bảo gấp nhưng đơn này thì không cần",

        "gấp là chuyện của đơn kia, đơn này cứ từ từ",

        "lần trước gấp, lần này thì thong thả",

        "không phải đơn gấp, cứ theo lịch",

        "đơn gấp xong rồi, còn đơn này thì không vội",

        "đừng chạy nhanh quá, đơn này chưa gấp",

    ],

    ("fragile", "yes", "std"): [

        "hàng {w}, nhớ nhẹ tay",

        "món này thuộc loại {w}, vận chuyển cẩn thận giúp {me}",

        "nhẹ tay nhé, đây là đồ {w}", "lưu ý: hàng {w}",

        "đừng xóc mạnh vì món này {w}", "kiện này {w}, bảo quản cẩn thận nha",

        "xử lý nhẹ tay một chút, hàng khá {w}",

        "đồ {w} đó, xếp riêng giúp {me}", "hàng dạng {w} nên xếp trên cùng",

        "bên trong là đồ {w}", "cẩn thận vì hàng khá {w}",

        "dán nhãn hàng {w} giúp {me}", "{o} thuộc loại {w}",

    ],

    ("fragile", "no", "std"): [

        "món này không {w}", "đây không phải hàng {w}",

        "không có lưu ý gì về đồ {w}", "kiện này không thuộc loại {w}",

        "hàng thường, không {w}", "không cần xem món này là đồ {w}",

        "hàng chắc chắn, không {w}",

        "đóng gói thường thôi, không phải đồ {w}", "không có gì {w} hết",

        "hàng không {w}, cứ xếp bình thường",

    ],

    ("fragile", "yes", "hard"): [

        "đừng làm rơi", "không được va đập",

        "tuyệt đối không xóc mạnh vì đồ {w}", "đừng quăng, đồ {w} đó",

        "đừng để bị đè lên vì {w}", "không được chồng đồ lên trên, hàng {w}",

    ],

    ("fragile", "no", "hard"): [

        "đơn của {name} mới {w}, đơn này thì không",

        "hôm trước gửi đồ {w}, lần này hàng thường",

        "đừng nhầm với đơn {w} kia",

        "không phải đồ {w} đâu, cứ xếp bình thường",

        "{w} là đơn kia, đơn này thì khác",

        "đơn {w} đã đi trước rồi, đơn này là hàng thường",

    ],

    # diễn đạt không dùng từ trong BANK['fragile'] (tắt bằng use_para_attr=False)

    ("fragile", "yes", "para"): [

        "đừng chồng đồ nặng lên trên", "tránh va đập mạnh", "đừng ném hay quăng",

        "cầm nhẹ tay thôi", "đặt cẩn thận, tránh rung lắc", "nhẹ tay, kẻo bể",

        "xếp riêng ra, đừng để đè",

    ],

    ("fragile", "no", "para"): [

        "đồ cứng cáp, không sợ va chạm", "hàng bình thường, xử lý như mọi khi",

        "hàng chắc chắn, xếp thoải mái",

        "đóng gói thường thôi, chồng đồ cũng được",

        "không cần giữ gìn đặc biệt", "cứ xếp như hàng bình thường",

    ],

}



ATTR_COMB = {

    ("yes", "yes"): [

        "vừa gấp vừa {w}, ráng giúp {me}",

        "gấp mà lại {w}, đi nhanh nhưng nhẹ tay", "đơn gấp và là hàng {w}",

        "nhanh thì nhanh nhưng nhớ nhẹ tay, đồ {w}",

        "hàng {w} nên nhẹ tay, mà {me} cũng cần gấp",

    ],

    ("yes", "no"): [

        "gấp nhưng không {w}", "đơn gấp, còn hàng thì không {w}",

        "cần nhanh, hàng không {w} nên khỏi lo", "gấp chứ không phải hàng {w}",

        "ưu tiên đơn này, hàng thường thôi",

        "chạy nhanh giúp {me}, đồ không {w}",

    ],

    ("no", "yes"): [

        "không gấp, chỉ cần nhẹ tay vì hàng {w}", "hàng {w} nhưng không vội",

        "từ từ cũng được, miễn là hàng {w} được giữ cẩn thận",

        "đồ {w}, đi chậm mà chắc", "chưa cần gấp, nhớ là hàng {w}",

        "đừng vội, hàng {w} mà",

    ],

    ("no", "no"): [

        "không gấp, cũng không {w}", "đơn thường, không vội, không phải hàng {w}",

        "hàng bình thường và không cần ưu tiên", "không gấp mà cũng chẳng {w}",

        "thong thả thôi, hàng thường",

    ],

}



FILL_PRE = ["chào {you}", "hi {you}", "alo", "{you} ơi", "cho {me} hỏi chút",

            "ok {you}", "dạ", "ừ", "à", "này", "đơn mới nè",

            "chào {you}, {me} gửi thông tin đơn nhé"]

FILL_POST = ["cảm ơn {you} nhiều", "thanks nha", "cảm ơn nhé", "ok chốt vậy nhé",

             "có gì nhắn {me} nha", "xong báo {me} một tiếng", "{me} đợi tin nhé",

             "cảm ơn {you} ạ", "cám ơn nhiều", "chốt vậy nha"]

FILL_INFO = ["mã đơn #{k}", "đơn đặt lúc {h}h", "khách tên {name_short}",

             "sđt cuối {d4}", "đơn đã thanh toán rồi", "đơn thu hộ COD",

             "khách đã xác nhận rồi", "trời đang mưa nhỉ",

             "hôm nay đông khách quá", "đơn số {k}"]





# ---------------------------------------------------------------

# 2. BALANCER (giữ nguyên từ V2)

# ---------------------------------------------------------------

class Balancer:

    def __init__(self, rng):

        self.rng = rng

        self.c = collections.Counter()



    def pick(self, key, options):

        opts = list(dict.fromkeys(options))

        if not opts:

            raise ValueError(f"Không có option để pick: {key!r}")

        m = min(self.c[(key, o)] for o in opts)

        choices = [o for o in opts if self.c[(key, o)] == m]

        o = self.rng.choice(choices)

        self.c[(key, o)] += 1

        return o





# ---------------------------------------------------------------

# 3. TEXT / NOISE HELPERS

# ---------------------------------------------------------------

def cap(s):

    return s[:1].upper() + s[1:] if s else s





def first_upper(text):

    return cap(text)





_TONES = {"\u0301": "s", "\u0300": "f", "\u0309": "r", "\u0303": "x", "\u0323": "j"}





def to_telex(word):

    """'chuyển' -> 'chuyeenr' : lỗi gõ khi bộ gõ tiếng Việt chưa bật."""

    out, tone = [], ""

    for ch in unicodedata.normalize("NFD", word):

        if ch in _TONES:

            tone = _TONES[ch]

        elif ch == "\u0302" and out:

            out.append(out[-1])

        elif ch in ("\u031b", "\u0306"):

            out.append("w")

        elif ch == "đ":

            out.append("dd")

        elif ch == "Đ":

            out.append("DD")

        else:

            out.append(ch)

    return "".join(out) + tone





def _build_kb():

    rows = ["qwertyuiop", "asdfghjkl", "zxcvbnm"]

    nb = {}

    for ri, row in enumerate(rows):

        for ci, ch in enumerate(row):

            s = []

            if ci > 0:

                s.append(row[ci - 1])

            if ci < len(row) - 1:

                s.append(row[ci + 1])

            for rj in (ri - 1, ri + 1):

                if 0 <= rj < len(rows) and ci < len(rows[rj]):

                    s.append(rows[rj][ci])

            nb[ch] = s

    return nb





_KB = _build_kb()





def _mutate_word(core, rng):

    op = rng.choices(["swap", "drop", "dup", "kb", "telex", "elong"],

                     weights=[0.28, 0.22, 0.12, 0.16, 0.14, 0.08], k=1)[0]

    n = len(core)

    if op == "telex":

        t = to_telex(core)

        return t if t != core else core + core[-1]

    if op == "elong":

        return core + core[-1] * rng.randint(1, 2)

    pos = rng.randint(1, n - 2)

    if op == "swap":

        return core[:pos] + core[pos + 1] + core[pos] + core[pos + 2:]

    if op == "drop":

        return core[:pos] + core[pos + 1:]

    if op == "dup":

        return core[:pos] + core[pos] + core[pos:]

    idxs = [i for i, ch in enumerate(core) if ch.lower() in _KB]

    if not idxs:

        return core

    i = rng.choice(idxs)

    return core[:i] + rng.choice(_KB[core[i].lower()]) + core[i + 1:]





def add_typo(text, rng):

    """Typo đa dạng: swap/drop/dup/phím lân cận/Telex quên bỏ dấu/kéo dài/dính chữ.

    Bỏ qua từ chứa '@' (placeholder địa điểm)."""

    if not text:

        return text

    words = text.split()

    cand = [i for i, w in enumerate(words)

            if "@" not in w and not re.search(r"\d", w)

            and len(re.sub(r"[^\w]", "", w)) >= 4]

    if not cand:

        return text

    n = rng.choices([1, 2, 3], weights=[0.62, 0.28, 0.10], k=1)[0]

    rng.shuffle(cand)

    for idx in cand[:n]:

        m = re.match(r"^(\W*)(.*?)(\W*)$", words[idx], flags=re.UNICODE)

        pre, core, post = m.group(1), m.group(2), m.group(3)

        if len(core) < 4:

            continue

        words[idx] = pre + _mutate_word(core, rng) + post

    if rng.random() < 0.10 and len(words) > 3:      # dính chữ (thiếu space)

        i = rng.randrange(len(words) - 1)

        if "@" not in words[i] and "@" not in words[i + 1]:

            words[i] = words[i] + words[i + 1]

            del words[i + 1]

    return " ".join(words)





_ABBR = {

    "không": ["ko", "k", "hok", "hông", "khong"], "được": ["đc", "dc"],

    "với": ["vs"], "bạn": ["bn"], "mình": ["mk", "mik"], "rồi": ["r", "rùi"],

    "luôn": ["lun"], "thôi": ["thui"], "vậy": ["zậy"], "nhưng": ["nhg"],

    "xong": ["xg"], "gửi": ["gởi"], "ok": ["oke", "okie"], "cảm": ["cám"],

    "người": ["ng"], "nhé": ["nhe"],

}

_ELONG = {"nhé", "nha", "với", "ạ", "nhen", "nghen", "nè", "nhe"}





def chat_noise(text, rng):

    """Văn phong nhắn tin: viết tắt, kéo dài particle, bỏ dấu phẩy."""

    toks = re.findall(r"\S+|\s+", text)

    out = []

    for t in toks:

        if t.isspace() or "@" in t:

            out.append(t)

            continue

        m = re.match(r"^([^\w]*)(.*?)([^\w]*)$", t, flags=re.UNICODE)

        pre, core, post = m.group(1), m.group(2), m.group(3)

        lc = core.lower()

        if lc in _ABBR and rng.random() < 0.55:

            core = rng.choice(_ABBR[lc])

        elif lc in _ELONG and rng.random() < 0.25:

            core = core + core[-1] * rng.randint(1, 3)

        out.append(pre + core + post)

    s = "".join(out)

    if rng.random() < 0.30:

        s = s.replace(",", "")

    return s





def norm_ws(txt):

    txt = re.sub(r"\s+", " ", txt)

    txt = re.sub(r"\s+([,.;!?:])", r"\1", txt)

    txt = re.sub(r"([,;:])\1+", r"\1", txt)

    txt = re.sub(r",\s*([.!?])", r"\1", txt)

    return txt.strip(" ,;")





def sentence(u, rng, p_excl=0.07):

    u = re.sub(r"\s+", " ", u).strip()

    if not u:

        return u

    u = cap(u)

    if u[-1] not in ".!?…)\"":

        u += "!" if rng.random() < p_excl else "."

    return u





_RECAP = re.compile(r"(?<![.])([.!?]\s+)([a-zà-ỹ])")





def recap(s):

    return _RECAP.sub(lambda m: m.group(1) + m.group(2).upper(), s)





def render_units(units, rng, style):

    units = [u for u in units if u and u.strip()]

    if style == "formal":

        return recap(" ".join(sentence(u, rng) for u in units))

    if style == "terse":

        sep = rng.choice([" | ", " / ", "; ", " - "])

        return sep.join(re.sub(r"[\s.;,]+$", "", u.strip()) for u in units)

    sep = rng.choice([", ", ", ", ". ", " - ", " . ", " "])

    parts = [re.sub(r"[\s.;,]+$", "", u.strip()).lower() for u in units]

    return sep.join(parts)





# ---------------------------------------------------------------

# 4. ĐỊA ĐIỂM: ALIAS / CLASH / PHRASE

# ---------------------------------------------------------------

_ALIAS_CACHE = {}

_CLASH = {}





def _alias_set(p):

    s = _ALIAS_CACHE.get(p)

    if s is None:

        s = {fold(a).lower().strip() for a in BANK.get(p, [p])}

        _ALIAS_CACHE[p] = s

    return s





def clash(p, q):

    """Hai địa điểm có chung alias -> không được dùng làm goal/distractor của nhau

    (nếu không, label sẽ mơ hồ)."""

    if p == q:

        return True

    k = (p, q) if p < q else (q, p)

    v = _CLASH.get(k)

    if v is None:

        v = bool(_alias_set(p) & _alias_set(q))

        _CLASH[k] = v

    return v





def _distract(B, key, forb):

    opts = [p for p in PLACES if not any(clash(p, f) for f in forb)]

    if not opts:

        opts = [p for p in PLACES if p not in forb] or list(PLACES)

    return B.pick(key, opts)





def _relation(rng, B, raw, role):

    if rng.random() >= CFG["p_kind"] or not KINDS:

        return "none", "none"

    kind = B.pick((role, "kind"), KINDS)

    if kind not in ("near", "far"):

        return kind, "none"

    opts = [p for p in PLACES if not clash(p, raw)] or [p for p in PLACES if p != raw]

    return kind, B.pick((role, "anchor"), opts)





def alias_for(raw_place, role, B, has_typo, rng):

    aliases = BANK.get(raw_place, [raw_place])

    value = B.pick(("place_alias", role, raw_place), aliases)

    if has_typo and rng.random() < CFG["p_alias_typo"]:

        value = add_typo(value, rng)

    return value





# Các dạng này CHỈ dùng kind_word làm từ quan hệ (không hardcode 'gần'

# như V2 -> tránh "xa X, gần Y" khi kind='far').

KIND_ONLY_FORMS = ["{base} {k}", "{k} {base}", "{base}, {k}", "{base} ({k})",

                   "chỗ {base} {k}", "{base} — {k}"]

ANCHOR_FORMS = ["{base} {k} {a}", "{base}, {k} {a}", "{base} ({k} {a})",

                "{base}, chỗ {k} {a}", "{k} {a}, chỗ {base}",

                "{base} nằm {k} {a}", "{base} — {k} {a}",

                "chỗ {base} ấy, {k} {a}", "{base} {k} {a} đó"]





def place_phrase(raw_place, kind="none", anchor="none", role="place", B=None,

                 has_typo=False, rng=None):

    rng = rng or random.Random()

    B = B or Balancer(rng)

    base = alias_for(raw_place, role, B, has_typo, rng)

    if kind == "none":

        return base

    k = B.pick(("kind_alias", kind), BANK.get(kind, [kind]))

    if anchor == "none":

        return B.pick(("kind_form", role, kind), KIND_ONLY_FORMS).format(base=base, k=k)

    a = alias_for(anchor, f"anchor_{role}", B, has_typo, rng)

    return B.pick(("kind_anchor_form", role, kind), ANCHOR_FORMS).format(base=base, k=k, a=a)





# ---------------------------------------------------------------

# 5. CTX: bộ sinh mệnh đề (ghép từ thành phần)

# ---------------------------------------------------------------

class _Fmt(dict):

    def __init__(self, base, lazy):

        super().__init__(base)

        self._lazy = lazy



    def __missing__(self, k):

        if k in self._lazy:

            v = self._lazy[k]()

            self[k] = v

            return v

        raise KeyError(k)





def render_vp(P, place, with_obj=True):

    o = (" " + P["o"]) if (with_obj and P["o"]) else ""

    return f"{P['v']}{P['adv']}{o} {P['p']} {place}"





class Ctx:

    def __init__(self, rng, B, bk, persona, item):

        self.rng, self.B, self.bk = rng, B, bk

        self.me, self.you, self.south, self.polite = persona

        self.item = item



    # ---- thành phần nhỏ ----

    def tail(self, mid=False, p_none=0.5):

        r = self.rng

        if r.random() < p_none:

            return ""

        pool = list(TAIL_COMMON) + (TAIL_SOUTH if self.south else TAIL_NORTH)

        if self.polite:

            pool += TAIL_POLITE

        if not mid:

            pool += TAIL_END_ONLY

        return r.choice(pool).format(me=self.me)



    def obj(self, allow_empty=False):

        r = self.rng

        if allow_empty and r.random() < 0.22:

            return ""

        x = r.random()

        if x < 0.38:

            return self.item

        if x < 0.50:

            return "đơn " + self.item

        if x < 0.58:

            return "món " + self.item

        return r.choice(OBJ_PRON)



    def vp_parts(self):

        B, r, bk = self.B, self.rng, self.bk

        verb = B.pick((bk, "verb"), list(GIVE_VERBS))

        prep = B.pick((bk, "prep", verb), GIVE_VERBS[verb])

        adv = ""

        if verb != "để lại" and r.random() < 0.16:

            adv = " " + r.choice(["thẳng", "luôn", "trực tiếp"])

        return {"v": verb, "p": prep, "adv": adv, "o": self.obj(allow_empty=True)}



    def vp(self, place):

        return render_vp(self.vp_parts(), place)



    def vp_contrast(self, G, T):

        P = self.vp_parts()

        fG, sG = render_vp(P, G), render_vp(P, G, False)

        fT, sT = render_vp(P, T), render_vp(P, T, False)

        p = P["p"]

        forms = [

            f"{fG}, chứ không {sT}", f"không {sT} mà {fG}",

            f"{fG} thôi, đừng {sT}", f"đừng {fT}; {sG} mới đúng",

            f"{sG}, không phải {p} {T}", f"đừng {fT}, hãy {sG}",

            f"{fG} nhé, đừng nhầm sang {T}",

            f"{P['v']}{P['adv']} {P['o']} không phải {p} {T} mà là {p} {G}",

        ]

        return self.B.pick((self.bk, "contrast"), forms)



    def gs(self, G, composable=True, family=None):

        r, me, you = self.rng, self.me, self.you

        fams = GS_FAMS if composable else GS_FAMS + ["ask"]

        fam = family or self.B.pick((self.bk, "gs_fam", composable), fams)

        t = self.tail()

        if fam == "imp":

            s = f"{self.vp(G)} {t}"

        elif fam == "nho":

            s = f"{r.choice(['nhờ', 'phiền', 'xin nhờ', 'làm phiền'])} {you} {self.vp(G)} {t}"

        elif fam == "you":

            s = f"{you} {self.vp(G)} {t}"

        elif fam == "want":

            s = r.choice([f"{me} muốn nhờ {you} {self.vp(G)}",

                          f"{me} cần {you} {self.vp(G)}",

                          f"{me} muốn {self.vp(G)}"]) + " " + t

        elif fam == "pass":

            P = self.vp_parts()

            aux = r.choice(["cần được", "phải được", "muốn được", "sẽ được"])

            s = f"{self.obj(False)} {aux} {P['v']}{P['adv']} {P['p']} {G} {t}"

        elif fam == "topic":

            P = self.vp_parts()

            s = f"{self.obj(False)}{r.choice([' thì', ''])} {render_vp(P, G, False)} {t}"

        elif fam == "formal":

            lead = r.choice(["vui lòng", "đề nghị", "yêu cầu", "kính nhờ " + you, "xin vui lòng"])

            s = f"{lead} {self.vp(G)}"

        elif fam == "nominal":

            s = r.choice(NOMINAL).format(G=G)

        elif fam == "recv":

            s = r.choice(RECV).format(G=G)

        elif fam == "frag":

            s = r.choice(FRAG).format(G=G, t=t)

        elif fam == "go":

            s = r.choice(GO).format(G=G, me=me, t=t)

        else:  # ask

            s = r.choice([f"có thể {self.vp(G)} không {you}?",

                          f"{you} {self.vp(G)} giúp {me} được không?",

                          f"{self.vp(G)} được không?"])

        return re.sub(r"\s+", " ", s).strip()



    def pv(self, V):

        r, B = self.rng, self.B

        if r.random() < 0.12:

            return B.pick((self.bk, "stop"), STOP_PATTERNS).format(V=V)

        pat = B.pick((self.bk, "pick"), PICK_PATTERNS)

        o = r.choice(PICK_OBJ).format(item=self.item)

        if "nhận" in pat and not o:

            o = "hàng"

        return re.sub(r"\s+", " ", pat.format(o=o, V=V)).strip()



    def ps(self, V):

        tpl = self.B.pick((self.bk, "ps"), PS_FRAMES)

        return re.sub(r"\s+", " ", self.fmt(tpl, V=V)).strip()



    def redir(self, G):

        return self.fmt(self.B.pick((self.bk, "redir"), REDIR_FORMS), G=G)



    def qa(self):

        q = self.fmt(self.B.pick((self.bk, "qa_q"), QA_Q))

        a = self.fmt(self.B.pick((self.bk, "qa_a"), QA_A))

        r = self.rng

        x = r.random()

        if x < 0.25:

            return f"A: {q} B: {a}"

        if x < 0.35:

            return f"- {q} - {a}"

        return q + r.choice([" ", " ", " — ", " - "]) + a



    # ---- bộ khai triển template ----

    def fmt(self, tpl, **kw):

        G = kw.get("G", PH["G"])

        V = kw.get("V", PH["V"])

        T = kw.get("T", PH["T1"])

        B, r, bk = self.B, self.rng, self.bk

        base = dict(G=G, V=V, T=T, TV=PH["TV"], T1=PH["T1"], T2=PH["T2"],

                    T3=PH["T3"], T4=PH["T4"], me=self.me, you=self.you,

                    Me=cap(self.me), You=cap(self.you))

        base.update({k: v for k, v in kw.items() if k not in ("G", "V", "T")})

        nm = r.choice(NAMES)

        lazy = {

            "o": lambda: self.obj(False),

            "tail": lambda: self.tail(),

            "tailm": lambda: self.tail(mid=True),

            "name": lambda: nm,

            "name_short": lambda: nm.split()[-1],

            "n": lambda: r.choice([10, 15, 20, 30, 45]),

            "k": lambda: r.randint(100, 9999),

            "h": lambda: r.randint(6, 21),

            "d4": lambda: f"{r.randint(0, 9999):04d}",

            "GV": lambda: self.vp(G),

            "GS": lambda: self.gs(G),

            "GV_T": lambda: self.vp(T),

            "GS_T": lambda: self.gs(T),

            "GV_C": lambda: self.vp_contrast(G, T),

            "PV": lambda: self.pv(V),

            "PS": lambda: self.ps(V),

            "REDIR": lambda: self.redir(G),

            "corr": lambda: self.fmt(B.pick((bk, "corr"), CORR_SOFT)),

            "Corr": lambda: cap(self.fmt(B.pick((bk, "corr"), CORR_SOFT))),

            "not_T": lambda: self.fmt(B.pick((bk, "not_T"), NOT_T), T=T),

            "notT_lead": lambda: self.fmt(B.pick((bk, "notT_lead"), NOT_LEAD), T=T),

            "why": lambda: B.pick((bk, "why"), WHY),

            "conv": lambda: B.pick((bk, "conv"), CONV),

        }

        return tpl.format_map(_Fmt(base, lazy))





# ---------------------------------------------------------------

# 6. THUỘC TÍNH (urgent / fragile)

# ---------------------------------------------------------------

def one_attr(ctx, which, val, w):

    r = ctx.rng

    kind, x = "std", r.random()

    if x < CFG["p_attr_hard"]:

        kind = "hard"

    elif (CFG["use_para_attr"] and which == "fragile"

          and x < CFG["p_attr_hard"] + CFG["p_attr_para"]):

        kind = "para"

    tpl = ctx.B.pick((ctx.bk, "attr", which, val, kind), ATTR[(which, val, kind)])

    return ctx.fmt(tpl, w=w)





def attr_units(ctx, urgent, fragile, w):

    r = ctx.rng

    omit_u = urgent == "no" and r.random() < CFG["p_omit_no"]

    omit_f = fragile == "no" and r.random() < CFG["p_omit_no"]

    if not omit_u and not omit_f and r.random() < CFG["p_attr_combined"]:

        tpl = ctx.B.pick((ctx.bk, "attr_comb", urgent, fragile), ATTR_COMB[(urgent, fragile)])

        return [ctx.fmt(tpl, w=w)], "combined"

    units = []

    if not omit_u:

        units.append(one_attr(ctx, "urgent", urgent, w))

    if not omit_f:

        units.append(one_attr(ctx, "fragile", fragile, w))

    return units, ("omit" if (omit_u or omit_f) else "separate")





# ---------------------------------------------------------------

# 7. TẠO 1 MẪU

# ---------------------------------------------------------------

def _weighted(table, allowed, exclude):

    return [(k, w) for k, w in table.items()

            if (allowed is None or k in allowed) and k not in exclude]





def _make_sample(rng, B, bk, is_neg, has_acc, has_typo, allowed, exclude):

    persona = B.pick((bk, "persona"), PERSONAS)

    item = B.pick((bk, "item"), ITEMS)

    ctx = Ctx(rng, B, bk, persona, item)



    # ---- thực thể ----

    goal = B.pick((bk, "goal"), PLACES)

    gk, ga = _relation(rng, B, goal, (bk, "grel"))

    via = "none"

    if rng.random() < CFG["p_via"]:

        opts = [p for p in PLACES if not clash(p, goal)]

        if opts:

            via = B.pick((bk, "via"), opts)

    if via == "none":

        vk, va = "none", "none"

    else:

        vk, va = _relation(rng, B, via, (bk, "vrel"))



    forb = {goal} | ({via} if via != "none" else set())

    t1 = _distract(B, (bk, "t1"), forb)

    t2 = _distract(B, (bk, "t2"), forb | {t1})

    t3 = _distract(B, (bk, "t3"), forb | {t1, t2})

    t4 = _distract(B, (bk, "t4"), forb | {t1, t2, t3})

    tv = _distract(B, (bk, "tv"), forb | {t1, t2, t3, t4})



    def trap(raw, role):

        if rng.random() < CFG["p_trap_rel"]:

            k, a = _relation(rng, B, raw, (bk, role + "_rel"))

            return place_phrase(raw, k, a, role, B, has_typo, rng)

        return alias_for(raw, role, B, has_typo, rng)



    surf = {0: place_phrase(goal, gk, ga, "goal", B, has_typo, rng),

            2: alias_for(tv, "tv", B, has_typo, rng),

            3: trap(t1, "t1"), 4: trap(t2, "t2"),

            5: alias_for(t3, "t3", B, has_typo, rng),

            6: alias_for(t4, "t4", B, has_typo, rng)}

    if via != "none":

        surf[1] = place_phrase(via, vk, va, "via", B, has_typo, rng)



    # ---- chọn kiểu goal-unit ----

    pool = _weighted(NEG_W if is_neg else AFF_W, allowed, exclude)

    if pool and (not is_neg or rng.random() < CFG["p_neg_goal"]):

        gtype = rng.choices([k for k, _ in pool], weights=[w for _, w in pool], k=1)[0]

    else:

        gtype = "plain"

    is_neg_type = gtype in NEG_W



    if gtype == "plain":

        goal_unit = ctx.gs(PH["G"], composable=False)

    elif gtype == "qa_reject":

        goal_unit = (ctx.fmt(B.pick((bk, "tpl", "qa_rev"), QA_REV))

                     if rng.random() < 0.50 else ctx.qa())

    else:

        goal_unit = ctx.fmt(B.pick((bk, "tpl", gtype), GOAL_TPL[gtype]))



    neg_units = 1 if is_neg_type else 0



    # ---- via ----

    via_unit, via_mode = "", "none"

    if via != "none":

        if gtype == "plain" and rng.random() < CFG["p_via_composite"]:

            goal_unit = ctx.fmt(B.pick((bk, "tpl", "composite"), COMPOSITE))

            via_mode = "composite"

        elif is_neg and ((not is_neg_type) or rng.random() < 0.35):

            via_unit = ctx.fmt(B.pick((bk, "tpl", "reject_via"), EXTRA_TPL["reject_via"]))

            via_mode, neg_units = "reject", neg_units + 1

        else:

            via_unit, via_mode = ctx.ps(PH["V"]), "plain"



    # ---- extras (câu lừa phụ, không chứa goal) ----

    extras, extra_kinds = [], []

    if is_neg:

        if neg_units == 0 or rng.random() < CFG["p_extra_neg"]:

            opts = ["route_no_stop"]

            if via == "none":

                opts.append("no_stop")

            elif via_mode != "reject":

                opts.append("reject_via")

            k = B.pick((bk, "extra_neg"), opts)

            extras.append(ctx.fmt(B.pick((bk, "tpl", k), EXTRA_TPL[k]), T=PH["T4"]))

            extra_kinds.append(k)

            neg_units += 1

    elif rng.random() < CFG["p_extra_aff"]:

        extras.append(ctx.fmt(B.pick((bk, "tpl", "route_pass"), EXTRA_TPL["route_pass"]),

                              T=PH["T4"]))

        extra_kinds.append("route_pass")



    # ---- attrs ----

    urgent = B.pick((bk, "urgent"), ["yes", "no"])

    fragile = B.pick((bk, "fragile"), ["yes", "no"])

    fra_w = B.pick((bk, "fra_w"), BANK.get("fragile", ["dễ vỡ"]))

    attrs, attr_mode = attr_units(ctx, urgent, fragile, fra_w)



    # ---- fillers ----

    pre = post = ""

    infos = []

    if rng.random() < CFG["p_filler_pre"]:

        pre = ctx.fmt(B.pick((bk, "fpre"), FILL_PRE))

    if rng.random() < CFG["p_filler_post"]:

        post = ctx.fmt(B.pick((bk, "fpost"), FILL_POST))

    if rng.random() < CFG["p_filler_info"]:

        infos.append(ctx.fmt(B.pick((bk, "finfo"), FILL_INFO)))



    # ---- sắp xếp: thứ tự goal/via + chèn các unit còn lại vào vị trí ngẫu nhiên ----

    core = [goal_unit] + ([via_unit] if via_unit else [])

    if via_unit and rng.random() < 0.5:

        core.reverse()

    body = list(core)

    for u in extras + attrs + infos:

        body.insert(rng.randint(0, len(body)), u)

    if pre:

        body.insert(0, pre)

    if post:

        body.append(post)



    r_style = rng.random()

    style = ("chat" if r_style < CFG["p_chat"]

             else "terse" if r_style < CFG["p_chat"] + CFG["p_terse"] else "formal")

    masked = render_units(body, rng, style)



    # ---- bất biến để label luôn khớp text ----

    ok = (PH["G"] in masked

          and ((via != "none") == (PH["V"] in masked))

          and not (via == "none" and PH["TV"] in masked)

          and "{" not in masked and "}" not in masked)

    if not ok:

        return None



    pos_g, pos_t = masked.find(PH["G"]), masked.find(PH["T1"])

    # ---- nhiễu ngữ cảnh (không đụng tới entity) ----

    if has_typo and rng.random() < CFG["p_sentence_typo"]:

        masked = add_typo(masked, rng)

    if style == "chat":

        masked = chat_noise(masked, rng)



    txt = re.sub(r"@(\d)@", lambda m: surf.get(int(m.group(1)), m.group(0)), masked)

    if "@" in txt:

        return None

    if rng.random() < CFG["p_lower"]:

        txt = txt.lower()

    if not has_acc:

        txt = fold(txt)

    txt = norm_ws(txt)



    y = {

        "goal": H2I["goal"][goal], "via": H2I["via"][via],

        "gkind": H2I["gkind"][gk], "ganchor": H2I["ganchor"][ga],

        "vkind": H2I["vkind"][vk], "vanchor": H2I["vanchor"][va],

        "urgent": H2I["urgent"][urgent], "fragile": H2I["fragile"][fragile],

    }

    meta = {"gtype": gtype, "via_mode": via_mode, "extras": extra_kinds,

            "attr_mode": attr_mode, "style": style, "persona": persona,

            "goal_first": (pos_t < 0 or pos_g < pos_t) if pos_g >= 0 else None,

            "has_distractor": gtype not in ("plain",) or bool(extras)}

    return txt, y, meta





# ---------------------------------------------------------------

# 8. GENERATOR CHÍNH (cùng chữ ký với V2 + tham số holdout)

# ---------------------------------------------------------------

GEN_META = {}





def generate_15k_dataset(seed=42, num_per_bucket=1875, dedupe=True,

                         goal_types=None, exclude_goal_types=()):

    """goal_types / exclude_goal_types: chọn/loại KIỂU câu để làm tập OOD.

       Kiểu aff: plain, self_correct, old_info, unavailable,

                 preference_override, other_order

       Kiểu neg: reject_goal, qa_reject, note_wrong, reject_multi

    """

    rng = random.Random(seed)

    B = Balancer(rng)

    allowed = set(goal_types) if goal_types else None

    exclude = set(exclude_goal_types or ())

    GEN_META.clear()



    buckets = {n: [] for n in [

        "Aff_Acc_NoTypo", "Aff_Acc_Typo", "Aff_NoAcc_NoTypo", "Aff_NoAcc_Typo",

        "Neg_Acc_NoTypo", "Neg_Acc_Typo", "Neg_NoAcc_NoTypo", "Neg_NoAcc_Typo"]}

    global_seen = set()

    bad = 0



    for bk in buckets:

        is_neg = bk.startswith("Neg")

        has_acc = "NoAcc" not in bk

        has_typo = "NoTypo" not in bk

        attempts, max_attempts = 0, max(num_per_bucket * 60, 20000)

        while len(buckets[bk]) < num_per_bucket:

            attempts += 1

            if attempts > max_attempts:

                raise RuntimeError(f"Không tạo đủ dữ liệu unique cho {bk}: "

                                   f"{len(buckets[bk])}/{num_per_bucket}.")

            res = _make_sample(rng, B, bk, is_neg, has_acc, has_typo, allowed, exclude)

            if res is None:

                bad += 1

                continue

            txt, y, meta = res

            if dedupe and txt in global_seen:

                continue

            global_seen.add(txt)

            GEN_META[txt] = meta

            buckets[bk].append({"text": txt, "y": y, "syn": True})

    if bad:

        print(f"  (bỏ {bad} mẫu vi phạm bất biến label/text)")

    return buckets, B





# ---------------------------------------------------------------

# 9. AUDIT: đo độ đa dạng + cân bằng vị trí goal/distractor

# ---------------------------------------------------------------

def audit_dataset(buckets, top=5):

    texts = [it["text"] for l in buckets.values() for it in l]

    alias = sorted({a for p in PLACES for a in BANK.get(p, [p])} |

                   {a for k in (KINDS or []) for a in BANK.get(k, [k])},

                   key=len, reverse=True)

    rx = re.compile("|".join(re.escape(a) for a in alias), re.IGNORECASE) if alias else None

    sk = collections.Counter()

    for t in texts:

        s = rx.sub("<P>", t) if rx else t

        sk[re.sub(r"\d+", "<N>", s)] += 1

    n = len(texts)

    print(f"Tổng mẫu: {n} | skeleton khác nhau: {len(sk)} ({len(sk)/n:.1%})")

    print(f"Skeleton phổ biến nhất chiếm: {sk.most_common(1)[0][1]/n:.3%}")

    gt = collections.Counter(GEN_META[t]["gtype"] for t in texts if t in GEN_META)

    print("Phân bố kiểu goal-unit:", dict(gt))

    gf = [GEN_META[t]["goal_first"] for t in texts

          if t in GEN_META and GEN_META[t]["has_distractor"]

          and GEN_META[t]["goal_first"] is not None]

    if gf:

        print(f"Goal đứng TRƯỚC distractor: {sum(gf)/len(gf):.1%} (lý tưởng ~50%)")

    lens = [len(t.split()) for t in texts]

    print(f"Độ dài (từ): min {min(lens)}, TB {sum(lens)/n:.1f}, max {max(lens)}")





# # ================================================================


In [ ]:
# V3.2 chỉ bổ sung loại câu / từ vựng, không sửa generator V3 gốc
# ================================================================
# V3.2 ADDITIVE ONLY: extra semantic families, short aliases, urgent language
# Execute AFTER original Wiktionary cell AND AFTER unchanged V3 definitions,
# BEFORE any call to generate_15k_dataset().
# ================================================================
import collections, random, re

V32_PLACE_ALIASES = {
    'canteen': ['khu ẩm thực','bếp ăn','nhà bếp','quầy cơm','quầy ăn','quán cơm','quầy giải khát'],
    'clinic': ['phòng bác sĩ','phòng sơ cứu','trạm cứu thương','phòng cấp cứu','khu y tế'],
    'dorm': ['khu nội trú','nhà nội trú','nhà tập thể','khu ký túc','khu lưu trú'],
    'gate': ['chốt bảo vệ','trạm gác','cổng trường','cổng chính','cổng phụ','lối ra vào'],
    'lab': ['phòng thực nghiệm','phòng thực hành','xưởng chế tạo','khu thí nghiệm','phòng nghiên cứu'],
    'lecture': ['lớp học','giảng đường','khu giảng đường','phòng giảng','nhà học'],
    'library': ['phòng đọc','kho sách','khu học liệu','thư khố','quầy mượn sách'],
    'office': ['tòa hiệu bộ','nhà hiệu bộ','phòng giáo vụ','phòng hành chính','văn phòng khoa'],
    'parking': ['hầm xe','hầm để xe','bãi xe','nhà xe','bãi gửi xe','bãi đỗ xe'],
    'sports': ['nhà thi đấu','nhà thể chất','sân thể thao','sân bóng','sân thi đấu','khu tập luyện'],
}
for _cls, _words in V32_PLACE_ALIASES.items():
    _old = BANK.get(_cls, [])
    BANK[_cls] = list(dict.fromkeys(_old + _words))

# The user's Phase4 `_v3_gen(TRAIN_ALIASES, ...)` temporarily replaces place BANK.
# Therefore carry NEW words to its TRAIN_ALIASES, never into held-out P4_HOLD.
if 'TRAIN_ALIASES' in globals() and isinstance(TRAIN_ALIASES,dict):
    for _cls,_words in V32_PLACE_ALIASES.items():
        if _cls not in TRAIN_ALIASES:continue
        _held={fold(str(_w)).lower() for _w in (globals().get('P4_HOLD',{}).get(_cls,[]))}
        _pool=list(TRAIN_ALIASES[_cls])
        _pool.extend(_w for _w in _words if fold(_w).lower() not in _held)
        TRAIN_ALIASES[_cls]=list(dict.fromkeys(_pool))

# Add terms to the existing relation bank WITHOUT introducing new labels.
V32_REL_WORDS = {
 'near': ['gần', 'kề', 'sát', 'kề bên', 'kề cạnh', 'ngay cạnh', 'bên cạnh','liền kề','gần sát','cạnh bên'],
 'far': ['xa','cách xa','khá xa','xa tít','xa lắc','xa hẳn','ở xa','cách biệt'],
 'north': ['phía bắc','hướng bắc','mạn bắc','bên trên','phía trên'],
 'south': ['phía nam','hướng nam','mạn nam','bên dưới','phía dưới'],
 'east': ['phía đông','hướng đông','mạn đông','bên phải','phía phải'],
 'west': ['phía tây','hướng tây','mạn tây','bên trái','phía trái'],
 'urgent': ['hỏa tốc','cấp tốc','tức tốc','khẩn cấp','gấp rút','cấp bách','gấp gáp','lập tức','sát giờ'],
 'fragile': ['dễ vỡ','dễ bể','dễ gãy','dễ nứt','dễ hỏng','mong manh','đồ gốm','đồ sứ'],
}
for _cls, _words in V32_REL_WORDS.items():
    BANK[_cls] = list(dict.fromkeys(BANK.get(_cls, []) + _words))

# New controlled types; each template unequivocally identifies PH['G'] as final goal.
V32_NEW_GOAL_TPL = {
 'dispatch_update': [
  'lệnh điều phối vừa đổi: không giao {T} nữa, đích mới {G}',
  'bản cập nhật cuối ghi {G}, còn {T} là bản đã hủy',
  'đừng theo lộ trình cũ tới {T}, chuyển đơn này sang {G}',
  'điều phối xác nhận {G} mới là điểm nhận, bỏ {T}',
  'trên phiếu điều chỉnh ghi tới {G}, không phải {T}',
  'đơn đã được chuyển địa chỉ từ {T} sang {G}',
  'hủy chỉ dẫn tới {T}; thông báo mới yêu cầu {GV}',
  'bản ghi {T} hết hiệu lực rồi; {GS}',
 ],
 'recipient_override': [
  'hồi trước khách ở {T} nhưng hiện đang chờ ở {G}',
  'người nhận đã rời {T}, hiện tới {G} để nhận',
  'đối chiếu lại rồi, khách nhận tại {G}; {T} chỉ là nơi đã rời đi',
  'khách không còn ở {T}; điểm bàn giao là {G}',
  'xác nhận qua điện thoại: {G} mới là chỗ nhận, {T} bỏ qua',
  'người nhận báo lại rằng không ở {T} nữa, đang có mặt tại {G}',
 ],
 'cancel_and_replace': [
  'hủy yêu cầu giao {T}. Tạo lại đơn này với điểm đến {G}',
  'yêu cầu giao {T} bị hủy; thay vào đó {GS}',
  'thôi không đi {T}, chốt {G} làm nơi giao',
  'không thực hiện lệnh cũ tới {T}, hãy {GV}',
  'bỏ dòng giao tới {T}; dòng chính xác là giao {G}',
  'hủy điểm giao trước đó là {T}; phải giao tại {G}',
 ],
 'corrected_transcript': [
  'tin nhắn nhận dạng nhầm {T}, người gửi nói rõ là {G}',
  'máy ghi âm chép {T} nhưng địa điểm thật là {G}',
  'dòng trước bị gõ lỗi thành {T}, đọc đúng phải là {G}',
  'trong bản nháp viết {T}, lúc xác nhận sửa thành {G}',
  'đừng dựa vào chữ {T} nhận dạng sai; giao {G}',
 ],
 'two_destinations': [
  'kiện của {name} mang tới {T}, còn gói đang nói đến giao {G}',
  'danh sách ghi hai nơi: đơn kia {T}, đơn này nhận ở {G}',
  'khác đơn nhé: {T} dành cho kiện cũ, {G} là điểm giao hiện tại',
  'đồ của khách trước ở {T}; kiện này phải được đưa về {G}',
  'đơn hàng hiện tại không theo địa chỉ {T} của đơn trước mà tới {G}',
 ],
 'denied_hint': [
  'hệ thống gợi ý {T} nhưng không đúng; đích đã xác nhận là {G}',
  'ứng dụng đoán {T} là nơi nhận, người đặt hàng xác nhận {G}',
  'đừng dựa vào gợi ý {T}; {GS}',
  'chỉ dẫn tự động đánh dấu {T} sai, vẫn cần {GV}',
  'không chọn {T} theo máy, chọn {G} theo người gửi',
 ],
}
GOAL_TPL.update(V32_NEW_GOAL_TPL)
AFF_W.update({'dispatch_update':1.2,'recipient_override':1.2,'two_destinations':0.9})
NEG_W.update({'cancel_and_replace':1.3,'corrected_transcript':1.1,'denied_hint':1.1})

# Extend the SAME original pools; use only placeholders already understood by Ctx.fmt.
PICK_PATTERNS.extend([
 'tới {V} nhận {o}', 'trước hết qua {V} lấy {o}',
 'ghé {V} nhận lại {o}', 'đi {V} lấy {o} trước',
 'nhận bàn giao {o} tại {V}', 'qua {V} nhặt {o}',
 'đến {V} thu gom {o}', 'qua {V} lấy đủ {o}',
])
COMPOSITE.extend([
 'nhận {o} ở {V} trước khi giao tới {G}',
 'điểm xuất phát lấy hàng là {V}, điểm giao sau cùng là {G}',
 '{PV} xong rồi hãy {GV}',
 'chặng lấy hàng {PV}, chặng giao hàng {GV}',
 'không phải tới {G} lấy đồ; {PV} rồi mới {GV}',
])
EXTRA_TPL['route_pass'].extend([
 'đường có thể đi ngang {T} nhưng không giao tại đó',
 'thấy {T} dọc đường thì vẫn đi tiếp',
 'có thể nhìn thấy {T} lúc di chuyển, không phải điểm dừng',
])
EXTRA_TPL['route_no_stop'].extend([
 'đi ngang {T} không có nghĩa là lấy hàng ở đó',
 'đừng ghé {T}, tên đó chỉ dùng làm mốc',
 'gặp {T} trên đường thì cứ đi tiếp, không cần dừng',
])
EXTRA_TPL['reject_via'].extend([
 'điểm lấy hàng mới là {V}, {TV} đã bị đổi',
 'không nhận đồ từ {TV}; {PV}',
 'phiếu cũ ghi lấy ở {TV}, nhưng thực tế {PS}',
])
ATTR[('urgent','yes','std')].extend([
 'cần gấp trong ca này', 'ưu tiên xử lý ngay', 'đơn sắp quá giờ',
 'phải tới trước khi hết giờ', 'khách cần ngay lập tức',
 'không thể chờ thêm', 'đi gấp kẻo muộn', 'phải giao kịp giờ',
 'đang sát hạn giao', 'tình huống khẩn cấp', 'cần nhận ngay',
 'bên nhận đang giục', 'thời hạn còn rất ít', 'đơn giao hỏa tốc',
 'ưu tiên chuyển trước mọi đơn', 'khách hẹn nhận ngay bây giờ',
])
ATTR[('urgent','yes','hard')].extend([
 'đừng chậm phút nào nữa', 'không còn kịp nếu trì hoãn',
 'nếu trễ thêm sẽ lỡ hạn giao', 'đơn này đã đến hạn, phải giao ngay',
 'không có thời gian chờ', 'chuyến này phải nhanh nhất có thể',
])
ATTR[('urgent','no','std')].extend([
 'mai nhận cũng chưa muộn', 'không cần chen lịch',
 'đi đúng nhịp bình thường', 'khi rảnh hãy gửi',
 'không có hạn giao gấp', 'khách chưa cần nhận ngay',
 'có thể giao muộn hơn', 'không phải ưu tiên',
 'để sau cũng được', 'chưa tới hạn giao',
 'cứ hoàn thành các đơn khác trước',
])
ATTR[('urgent','no','hard')].extend([
 'đơn trước mới bị giục, đơn này chưa cần',
 'dòng thông báo gấp là của hôm qua, đơn này không gấp',
 'nghe chữ khẩn nhưng đây là lời nhắc cho đơn khác',
 'người nhận xác nhận chưa vội, không cần ưu tiên',
])
ATTR[('fragile','yes','std')].extend([
 'bên trong có đồ sứ, giữ nhẹ tay', 'hàng thủy tinh, tránh va đập',
 'kiện có đồ gốm, không được xóc', 'đồ nhạy với rung lắc',
 'bên trong dễ nứt, đừng làm rơi',
])
ATTR[('fragile','no','std')].extend([
 'hàng rất chắc, không lo vỡ', 'đây là đồ không cần chống sốc',
 'đóng gói như hàng thường', 'đồ bền, không phải đồ dễ vỡ',
])
# Keep original callbacks, tokenizer, loss, optimizer, scheduling and training UNMODIFIED.
_ALl = globals().get('_ALIAS_CACHE',None)
if isinstance(_ALl,dict): _ALl.clear()
if isinstance(globals().get('_CLASH',None),dict): _CLASH.clear()
print('V3.2 lexical + grammar add-on:', len(V32_NEW_GOAL_TPL), 'new goal families; original generator unchanged')


V3.2 lexical + grammar add-on: 6 new goal families; original generator unchanged


In [ ]:
VIA_TRICK_AFF = globals().get('VIA_TRICK_AFF', [])
# Mẫu câu không gian 5 nhãn mới, via và urgent
# SPATIAL+VIA+ANCHOR+URGENT EXTENSION, executed AFTER the V3 source definitions.
# 8-head goal=none for map-ref is mandatory (never use the raw map-resolved goal).
import random, re, hashlib, json, collections, unicodedata

# Improved V3 distributions from the new dataset (many spatial instances generated by V4 separately).
CFG.update({'p_kind':0.63, 'p_via':0.49, 'p_via_composite':0.25,
            'p_extra_aff':0.55, 'p_extra_neg':0.60, 'p_attr_hard':0.45,
            'p_attr_para':0.15, 'p_omit_no':0.10, 'p_sentence_typo':0.30,
            'p_alias_typo':0.28, 'p_via_trick':0.45, 'p_trap_rel':0.55})

GOAL_TPL.update({
 'fix_after_cancel':[
   'Hủy điểm {T} nhé, thay bằng {GS}', 'Không giao {T} nữa nhé, đổi lại: {GV}',
   'Bỏ yêu cầu đến {T}, chốt lại là {G}', 'Đơn cũ giao {T} đã hủy, đơn mới {GV}',
   'Địa chỉ {T} bị loại, {G} mới là điểm nhận', 'Đừng theo dòng ghi {T}, sửa thành {G}',
 ],
 'anchor_vs_goal':[
   '{GS}, còn {T} chỉ là mốc trên đường', 'Dù đứng cạnh {T} thì đích vẫn là {G}',
   'Khách nói về {T} để định hướng thôi, hàng vẫn {GV}',
   '{T} không phải nơi trả hàng, {GS}',
 ],
 'wrong_recipient':[
   'Khách ở {T} là của đơn khác, người nhận đơn này ở {G}',
   'Tôi đang đứng ở {T}, không phải người nhận, cứ {GV}',
   'Người nhận cũ rời {T} rồi, bên mới đợi ở {G}',
 ],
 'literal_vs_metaphor':[
   'Hồ sơ từ {T} chuyển tới, nhưng nơi giao thật là {G}',
   'Tập tài liệu nhắc {T}, đây chỉ là nội dung giấy tờ; {GS}',
   'Trên tem ghi {T} của lô cũ, đích thực tế: {G}',
 ]
})
AFF_W.update({'fix_after_cancel':1.9,'anchor_vs_goal':1.4,'wrong_recipient':1.1,'literal_vs_metaphor':0.7})
EXTRA_TPL['route_pass'] += [
    'đi ngang qua {T} thôi, không lấy hay giao gì cả',
    '{T} là điểm đi ngang chứ không phải nơi nhận',
    'đường có đi qua {T} thì cũng không dừng tại đó',
]
EXTRA_TPL['no_stop'] += [
    'trước đó có nhắc qua {T} lấy nhưng hàng đã có sẵn',
    'không lấy thêm gì ở {T} nhé, hàng đủ rồi',
]
EXTRA_TPL['reject_via'] += [
    'đơn kia lấy ở {TV}, nhưng lần này {PV}',
    'nơi lấy cũ là {TV}; bây giờ đổi sang {V}',
    'không tạt {TV} đâu, cứ {PV}',
]
VIA_TRICK_AFF += [
    'bỏ điểm lấy cũ {TV}, thay bằng {PV}',
    '{TV} không còn hàng, qua {V} nhận nhé',
    '{PV}; nơi {TV} chỉ của đơn đã hủy',
]
ATTR[('urgent','yes','std')] += [
    'hỏa tốc', 'ưu tiên số một, đi ngay', 'cần đến ngay', 'giao cấp tốc',
    'không được chậm trễ', 'phải tới kịp giờ', 'khách giục liên tục',
    'hàng cần gấp trong ca này', 'đơn ưu tiên cao',
    'trước giờ hẹn phải hoàn thành', 'giao thật nhanh',
    'khẩn cấp, xử lý liền', 'đẩy nhanh chuyến này',
]
ATTR[('urgent','no','std')] += [
    'chiều nay giao cũng được', 'mai mới cần hàng', 'không cần sớm quá',
    'đi đúng lịch là được', 'khách chưa cần ngay', 'không hỏa tốc đâu',
    'khỏi ưu tiên đơn này', 'không áp lực giờ giấc',
    'giao sau cũng được', 'lúc rảnh giao',
]
ATTR[('urgent','yes','hard')] += [
    'ban đầu khách chưa vội, nhưng vừa gọi giục gấp',
    'đã hẹn mai rồi đổi lịch sang ngay bây giờ',
    'trễ hẹn là không kịp ca trực',
    'đừng hiểu chữ gấp trong tài liệu, đơn này cũng đang rất gấp',
]
ATTR[('urgent','no','hard')] += [
    'khách bảo gấp giấy tờ lại, chứ hàng chưa cần giao ngay',
    'cứ gấp tập tài liệu lại, không phải giao gấp',
    'đơn khác giao hỏa tốc, còn đơn này đi thường',
    'không cần khẩn trương, ưu tiên đơn khác trước',
    'sáng báo khẩn cấp nhưng khách vừa hoãn sang mai',
]
ATTR[('fragile','yes','para')] += [
    'bên trong là đồ gốm, tránh va chạm',
    'thủy tinh bên trong, tránh lắc mạnh',
    'hàng dễ sứt mẻ, không được ném',
    'đồ sành có thể nứt nếu rung xóc',
]
ATTR[('fragile','no','para')] += [
    'hàng bằng nhựa cứng, không lo bể',
    'đồ mềm, không cần chống sốc',
    'thùng này không chứa đồ gốm hay thủy tinh',
]

# Core map-to-goal templates, split by exact template ID to avoid synthetic validation leakage.
MAP_TPL = {
 'anchor_near':[
    'địa điểm gần {A} nhất', 'nơi sát {A} nhất', 'chỗ kề {A} nhất',
    'địa điểm cạnh {A} nhất', 'vị trí liền kề {A} nhất',
    'chỗ ngay bên {A} nhất', 'chỗ gần {A} nhất trên bản đồ',
    'địa điểm có khoảng cách ngắn nhất tới {A}', 'nơi gần {A} hơn cả',
    'chỗ cách {A} ngắn nhất', 'nơi cách {A} ít bước nhất',
    'địa điểm không xa {A} nhất', 'địa điểm lân cận {A} nhất',
    'chỗ sát cạnh {A} nhất', 'địa điểm ở gần {A} nhất',
    'nơi cạnh bên {A} nhất', 'điểm gần kề {A} nhất',
    'chỗ có khoảng cách nhỏ nhất đến {A}',
    'nơi sát {A} hơn tất cả điểm khác',
    'điểm kề {A} hơn mọi nơi',
 ],
 'north_most':[
    'địa điểm ở trên cùng bản đồ', 'chỗ phía bắc nhất',
    'nơi nằm cao nhất bản đồ', 'địa điểm xa nhất về phía bắc',
    'chỗ sát mép trên nhất', 'điểm cực bắc',
    'nơi ở hướng bắc xa nhất', 'vị trí cao nhất trên sơ đồ',
    'địa điểm nằm trên cùng', 'chỗ ở mạn bắc xa nhất',
 ],
 'south_most':[
    'địa điểm ở dưới cùng bản đồ', 'chỗ phía nam nhất',
    'nơi nằm thấp nhất bản đồ', 'địa điểm xa nhất về phía nam',
    'chỗ sát mép dưới nhất', 'điểm cực nam',
    'nơi ở hướng nam xa nhất', 'vị trí thấp nhất trên sơ đồ',
    'địa điểm nằm dưới cùng', 'chỗ ở mạn nam xa nhất',
 ],
 'east_most':[
    'địa điểm bên phải nhất bản đồ', 'chỗ phía đông nhất',
    'nơi xa nhất về hướng đông', 'địa điểm sát mép phải nhất',
    'điểm cực đông', 'vị trí bên phải xa nhất',
    'chỗ nằm tận cùng phía đông', 'nơi ngoài cùng bên phải',
    'địa điểm xa nhất về phía đông', 'khu vực phía đông nhất',
 ],
 'west_most':[
    'địa điểm bên trái nhất bản đồ', 'chỗ phía tây nhất',
    'nơi xa nhất về hướng tây', 'địa điểm sát mép trái nhất',
    'điểm cực tây', 'vị trí bên trái xa nhất',
    'chỗ nằm tận cùng phía tây', 'nơi ngoài cùng bên trái',
    'địa điểm xa nhất về phía tây', 'khu vực phía tây nhất',
 ],
}
MAP_FRAMES = [
 'Giao {O} tới {G}', 'Mang {O} đến {G}',
 'Đích đến là {G}, hàng cần giao: {O}',
 'Bàn giao {O} tại {G}', 'Điểm nhận cuối cùng: {G}',
 'Chuyển {O} qua {G}', 'Đơn này đưa tới {G}',
 'Người nhận đợi ở {G}', 'Đưa kiện này đến {G} giúp mình',
 'Làm ơn đi đến {G} để giao hàng',
 'Điểm giao: {G}. Hàng: {O}',
 'Bên nhận đang chờ ở {G}',
]
VIA_FRAMES = [
 'ghé {V} lấy {O}', 'qua {V} nhận {O}',
 'tạt qua {V} lấy hàng', 'đến {V} nhận {O}',
 'lấy {O} ở {V}', 'nhận {O} tại {V}',
 'qua {V} bốc {O}', 'nhận {O} từ {V}',
]
MAP_COMB = [
 '{V}, rồi {G}', '{V}. Sau đó {G}',
 '{V}, tiếp đến {G}', '{G}; nhưng trước đó cần {V}',
 '{V}; chặng cuối {G}',
 '{G}. Điểm lấy hàng trước đó: {V}',
 '{V}. Sau cùng {G}',
 '{G}. Việc đầu tiên là {V}',
]
MAP_TRAPS = [
    'Không giao {T} nữa nhé, đổi lại: {G}',
    'Hôm qua giao tại {T} rồi, hôm nay {G}',
    'Dừng nhầm ở {T} lần trước, nay {G}',
    'Đừng nhầm với {T} nhé. {G}',
    'Người nhận đã rời {T} rồi, {G}',
    'Bỏ qua {T}, không phải ở đó. {G}',
    'Tin trước ghi {T} là nhầm. Đúng ra: {G}',
    'Chỗ {T} chỉ là mốc, không trả hàng. {G}',
    '{G}. {T} không phải điểm nhận',
    'Nếu sau này cần thì mới giao ở {T}, hôm nay {G}',
    'Khách vừa đổi ý, không nhận ở {T}; {G}',
]
MAP_EXTRA = [
    'Không cần ghé {T}, hàng đã có sẵn trên xe',
    'Đơn kia phải lấy hàng ở {T}, đơn này không',
    'Người gửi trước đó đứng ở {T}, nhưng giờ đã về',
    'Đi ngang qua {T} thôi, không cần dừng',
    'Tấm bản đồ có đánh dấu {T} nhưng đó không phải đích',
    'Tránh nhầm sang {T} vì đó là đơn đã hủy',
]
URGENT_YES = [
    'Khẩn cấp, cần đi ngay', 'Đơn hỏa tốc', 'Càng nhanh càng tốt',
    'Không được chậm trễ', 'Đến ngay giúp mình', 'Ưu tiên số một',
    'Khách đang giục', 'Sát hạn giao rồi', 'Phải kịp giờ',
    'Gấp rút xử lý', 'Giao ngay trong ca', 'Chạy gấp giúp',
    'Khách cần nhận tức thì', 'Đừng để trễ giờ',
    'Không phải là không gấp', 'Lúc trước không vội, giờ cần giao ngay',
    'Đơn gấp hôm nay chứ không phải đơn kia',
]
URGENT_NO = [
    'Không gấp', 'Từ từ cũng được', 'Mai giao cũng kịp',
    'Chiều nay giao cũng được', 'Không cần vội', 'Cứ thong thả',
    'Hôm sau cũng ổn', 'Chưa cần ngay', 'Đúng lịch thường là được',
    'Không hỏa tốc', 'Không ưu tiên đơn này',
    'Đơn gấp là của người khác, đơn này thì không',
    'Hồ sơ đã gấp làm ba, nhưng hàng thì không cần giao gấp',
    'Sáng báo gấp rồi khách dời lịch đến tuần sau',
]
FRAGILE_YES = [
    'Bên trong là đồ thủy tinh', 'Hàng dễ vỡ, nhẹ tay',
    'Đồ gốm bên trong, tránh va chạm', 'Hàng mỏng manh, chống sốc',
    'Lưu ý hàng dễ bể', 'Không được quăng vì dễ nứt',
    'Vận chuyển nhẹ nhàng', 'Đừng làm rơi kiện này',
    'Có kính trong hộp, giữ cẩn thận',
]
FRAGILE_NO = [
    'Hàng chắc chắn, không dễ vỡ', 'Hàng thường thôi',
    'Không phải đồ dễ bể', 'Đồ mềm, không sợ va chạm',
    'Hộp này không cần chống sốc', 'Đừng nhầm với kiện thủy tinh kia',
    'Đơn hôm qua dễ vỡ, còn đơn này là hàng thường',
]

for _name, _lst in list(MAP_TPL.items()) + [
    ('MAP_FRAMES',MAP_FRAMES),('VIA_FRAMES',VIA_FRAMES),('MAP_COMB',MAP_COMB),
    ('MAP_TRAPS',MAP_TRAPS),('MAP_EXTRA',MAP_EXTRA)]:
    assert len(_lst)==len(set(_lst)), f'Duplicate template: {_name}'

def _split_templates(lst, split, foldcount=5):
    # Deterministic template-ID split: each family contributes >=1 true held-out template.
    # A global modulo can by chance select zero examples from a small family -> leakage.
    unique=list(dict.fromkeys(lst))
    ranked=sorted(unique,key=lambda t: hashlib.sha256(t.encode('utf8')).hexdigest())
    n_hold=max(1,round(len(ranked)/foldcount)) if len(ranked)>1 else 0
    hold=set(ranked[:n_hold])
    pool=[t for t in unique if (t in hold if split=='holdout' else t not in hold)]
    assert pool, (split,len(lst))
    return pool

def _pick_template(rng, items, split):
    return rng.choice(_split_templates(items,split))

def _short_alias(rng, place):
    return rng.choice(BANK[place])

def _typo_safe(s,rng):
    # Corrupt only CONTENT WORDS; preserve logical operators (negation/near/far/most).
    skip={'không','chẳng','đừng','khỏi','nữa','nhất','sát','gần','xa','cạnh',
          'trước','sau','nhưng','hủy','đổi','nay','hôm','qua','gấp','vội','bể','vỡ'}
    tokens=re.split(r'(\s+)',s)
    opts=[i for i,t in enumerate(tokens) if t.isalpha() and len(t)>4 and vocab_key(t) not in skip]
    if not opts:return s
    j=rng.choice(opts);tok=tokens[j];op=rng.choice(['drop','swap','repeat'])
    k=rng.randrange(1,len(tok)-1)
    if op=='drop':tokens[j]=tok[:k]+tok[k+1:]
    elif op=='repeat':tokens[j]=tok[:k]+tok[k]+tok[k:]
    else:tokens[j]=tok[:k]+tok[k+1]+tok[k]+tok[k+2:] if k+1<len(tok) else tok
    return ''.join(tokens)

def v4_spatial_sample(seed,split='train',force_kind=None, force_anchor=None,
                      fixed_flags=None, fixed_via=None):
    rng=random.Random(int(seed))
    kind=force_kind or rng.choices(list(MAP_TPL),weights=[48,13,13,13,13],k=1)[0]
    anchor=(force_anchor if force_anchor is not None else rng.choice(PLACES)) if kind=='anchor_near' else 'none'
    mp=_pick_template(rng,MAP_TPL[kind],split)
    goal_phrase=mp.format(A=_short_alias(rng,anchor)) if kind=='anchor_near' else mp
    gcl=_pick_template(rng,MAP_FRAMES,split).format(G=goal_phrase,O=rng.choice(ITEMS))
    via=(fixed_via if fixed_via is not None else (rng.choice(PLACES) if rng.random()<0.52 else 'none'))
    vkind='none';vanchor='none';vcl=''
    if via!='none':
        vkind=rng.choices(['none','near','far','north','south','east','west'],weights=[60,12,10,5,5,4,4],k=1)[0]
        va=_short_alias(rng,via)
        if vkind in ('near','far'):
            opts=[q for q in PLACES if q!=via]
            vanchor=rng.choice(opts)
            kw=rng.choice(BANK[vkind])
            va=va+' '+kw+' '+_short_alias(rng,vanchor)
        elif vkind!='none':
            va=va+' '+rng.choice(BANK[vkind])
        vcl=_pick_template(rng,VIA_FRAMES,split).format(V=va,O=rng.choice(ITEMS))
    if vcl:
        fmt=_pick_template(rng,MAP_COMB,split)
        text=fmt.format(V=vcl,G=gcl)
    else:text=gcl
    trap=rng.choice([p for p in PLACES if p not in (anchor,via)])
    if rng.random()<0.61:
        # All distractors are explicitly canceled or contextualized.
        text=_pick_template(rng,MAP_TRAPS,split).format(G=text,T=_short_alias(rng,trap))
    if rng.random()<0.30:
        text+=' . '+_pick_template(rng,MAP_EXTRA,split).format(T=_short_alias(rng,trap))
    u,f=fixed_flags if fixed_flags is not None else (rng.choice(['yes','no']),rng.choice(['yes','no']))
    attr=[rng.choice(URGENT_YES if u=='yes' else URGENT_NO),
          rng.choice(FRAGILE_YES if f=='yes' else FRAGILE_NO)]
    # Status before OR after route: deliberately force context attention.
    if rng.random()<0.5:text=' . '.join(attr+[text])
    else:text=' . '.join([text]+attr)
    if rng.random()<0.15:text='Robot ơi, '+text
    if rng.random()<0.16:text+=' . Cảm ơn nhé'
    view='clean'  # mandatory four views are generated ONCE for every base sample in CELL 10
    if view in ('typo','noaccent_typo'):text=_typo_safe(text,rng)
    if view in ('noaccent','noaccent_typo'):text=vocab_fold(text)
    text=re.sub(r'\s+',' ',text).replace(' . ','. ').strip(' .')
    y={'goal':'none','gkind':kind,'ganchor':anchor,
       'via':via,'vkind':vkind,'vanchor':vanchor,'urgent':u,'fragile':f}
    assert (kind=='anchor_near')==(anchor!='none')
    assert (vkind in ('near','far'))==(vanchor!='none')
    return {'text':text,'y':{h:H2I[h][y[h]] for h in HEADS_EXPECTED},
            'origin':'synthetic_v4_'+split,'kind':kind,'template':mp,'view':view}

def v4_spatial_generate(n,split,seed=21011):
    seen=set();out=[];attempt=0
    while len(out)<n:
        r=v4_spatial_sample(seed+attempt*17,split=split)
        attempt+=1
        key=vocab_key(r['text'])
        if key in seen:continue
        seen.add(key);out.append(r)
        if attempt>n*16:raise RuntimeError(f'not enough unique V4 spatial rows: {len(out)}/{n}')
    return out

# Opt-in Wiktionary reader: DO NOT automatically import suggestions into training bank.
# Wiki must never expand a vague phrase with 'sinh viên' or 4+ syllables.
ENABLE_WIKI = False
WIKI_MAX_QUERIES = 100

def wiki_short_candidates(max_queries=WIKI_MAX_QUERIES):
    import time, requests
    from bs4 import BeautifulSoup
    from urllib.parse import quote
    out=[];seen=set();queries=0
    for klass,terms in WIKI_SEEDS.items():
        for term in terms:
            if queries>=max_queries:return out
            assert 2<=len(term.split())<=3
            key=(klass,term)
            if key in seen:continue
            seen.add(key);queries+=1
            url='https://vi.wiktionary.org/wiki/'+quote(term.replace(' ','_'))
            try:
                r=requests.get(url,timeout=6,headers={'User-Agent':'CampusCourierResearch/1.0 (educational)'})
                if r.status_code!=200:continue
                html=BeautifulSoup(r.text,'html.parser')
                # Only propose 2-3 syllable candidates. No semantic drift by recursive crawl.
                for heading in html.find_all(['h2','h3','h4']):
                    if 'đồng nghĩa' not in heading.get_text(' ',strip=True).lower():continue
                    nxt=heading.find_next(['ul','ol'])
                    if not nxt:continue
                    for li in nxt.find_all('li')[:14]:
                        ss=re.split(r'[;,(/]',li.get_text(' ',strip=True))[0].strip().lower()
                        if (2<=len(ss.split())<=3 and ss not in BAD_CROSS_CLASS
                            and all(c.isalpha() or c.isspace() or c in 'àáảãạâăđêôơư' for c in ss)):
                            out.append({'class':klass,'seed':term,'candidate':ss,'approved':False})
                time.sleep(.12)
            except Exception:continue
    return out

In [ ]:
# ============================================================================
# THÊM NGÂN HÀNG CÂU V3 + 100 CỤM SPATIAL (20 CỤM / LOẠI)
# Giữ nguyên 8 bucket V3, HEADS, H2I và checkpoint.
# Chạy SAU cell định nghĩa MAP_TPL, TRƯỚC cell sinh train_rows.
# ============================================================================

import collections

# ============================================================================
# 1. GOAL TEMPLATES
# ============================================================================

EXTRA_NEW_GOAL_TPL = {
    "last_minute_update": [
        "Tin nhắn giao đến {T} đã hết hiệu lực, bây giờ {GS}",
        "Đổi đích phút chót: không phải {T}, {GV}",
        "Đã thu hồi hướng dẫn đi {T}; đơn này {GV}",
        "Phần ghi {T} chỉ là dự thảo, {G} mới được xác nhận",
        "Cập nhật cuối của khách: {G}, không phải {T}",
        "Chốt sau cùng: {GS}; {T} gạch bỏ",
    ],

    "quoted_wrong": [
        "Dòng báo giá nhắc tới {T}, nhưng địa chỉ nhận thật là {G}",
        "Trong giấy tờ có chữ {T}, còn lộ trình giao thật thì {GV}",
        "Tiêu đề kiện hàng ghi {T}, khách nhận tại {G}",
        "Đừng đọc tên {T} trên nhãn thành điểm đến; {GS}",
        "Bảng thông tin cũ đề cập {T}, nhưng thực tế {GS}",
    ],

    "recipient_vs_sender": [
        "Người gửi đang ở {T}; người nhận ở {G}, {GV}",
        "Hàng xuất phát từ {T} nhưng phải giao về {G}",
        "{T} là chỗ gửi ban đầu, không phải nơi trả hàng; {GS}",
        "Địa chỉ liên hệ là {T}; địa chỉ giao của đơn này là {G}",
    ],
}

for _fam, _patterns in EXTRA_NEW_GOAL_TPL.items():
    GOAL_TPL.setdefault(_fam, []).extend(_patterns)

AFF_W.update({
    "last_minute_update": 1.3,
    "quoted_wrong": 0.9,
    "recipient_vs_sender": 1.3,
})

# ============================================================================
# 2. VIA TEMPLATES
# ============================================================================

PICK_PATTERNS.extend([
    "ghé {V} để nhận {o}",
    "qua {V} lấy cho đủ {o}",
    "nhận kiện {o} từ {V}",
    "đến {V} bốc {o} lên",
    "trước hết nhận {o} tại {V}",
])

STOP_PATTERNS.extend([
    "qua {V} xử lý khâu lấy hàng",
    "điểm ghé đầu là {V}",
    "cần dừng ở {V} trước khi giao",
])

EXTRA_TPL["reject_via"].extend([
    "đã chuyển điểm lấy hàng từ {TV} sang {V}",
    "mốc {TV} chỉ là điểm lấy cũ, {PS}",
    "đừng lấy nhầm ở {TV} vì {V} mới có hàng",
])

# ============================================================================
# 3. URGENT / FRAGILE TEMPLATES
# ============================================================================

ATTR[("urgent", "yes", "std")].extend([
    "khách cần kiện này lập tức",
    "không kịp nếu giao theo lịch thường",
    "đây là đơn ưu tiên khẩn",
    "đến trước giờ đóng cửa giúp mình",
    "giao ngay trong ca trực",
    "làm gấp theo yêu cầu bên nhận",
    "phải bàn giao tức tốc",
    "giao càng sớm càng đỡ lỡ hẹn",
])

ATTR[("urgent", "no", "hard")].extend([
    "chữ gấp trên giấy là chỉ việc gấp giấy, không phải giao gấp",
    "khách vừa dời lịch sang tuần sau, không phải chuyển ngay",
    "đơn số trước cần gấp còn đơn này cứ đi theo lịch",
    "đã hết hạn khẩn cấp, hiện tại không cần giao vội",
])

ATTR[("urgent", "yes", "hard")].extend([
    "ban đầu chưa gấp, giờ khách đổi sang cần ngay",
    "mặc dù hệ thống ghi đơn thường, người nhận vừa yêu cầu gấp",
    "trễ là lỡ buổi nhận hàng, cần giao ngay",
])

ATTR[("fragile", "yes", "para")].extend([
    "có chai lọ thủy tinh, chớ làm xóc",
    "hộp chứa đồ sành sứ nên tránh rơi",
    "đóng gói nhẹ tay để không làm nứt",
])

ATTR[("fragile", "no", "hard")].extend([
    "bên nhận nói dễ vỡ là đơn hôm qua, gói này bằng vải",
    "phiếu cũ in fragile, hàng lần này là giấy tờ thường",
])

# ============================================================================
# 4. SPATIAL — 20 CỤM NGẮN, MỖI CỤM ĐÚNG 2 TIẾNG
# ============================================================================

MAP_SPATIAL_ALIASES = {

    "anchor_near": [
        "gần nhất",
        "sát nhất",
        "gần sát",
        "ngay cạnh",
        "sát cạnh",
        "sát bên",
        "sát vách",
        "kế bên",
        "kề bên",
        "cạnh bên",
        "liền kề",
        "gần kề",
        "cận kề",
        "ngay sát",
        "ngay bên",
        "kế cận",
        "gần hơn",
        "sát hơn",
        "sát nút",
        "cạnh đó",
    ],

    "north_most": [
        "bắc nhất",
        "cực bắc",
        "tận bắc",
        "mép bắc",
        "rìa bắc",
        "phía bắc",
        "hướng bắc",
        "mạn bắc",
        "bên bắc",
        "đầu bắc",
        "cuối bắc",
        "ngoài bắc",
        "trên cùng",
        "trên nhất",
        "cao nhất",
        "cực trên",
        "mép trên",
        "rìa trên",
        "đỉnh trên",
        "tận trên",
    ],

    "south_most": [
        "nam nhất",
        "cực nam",
        "tận nam",
        "mép nam",
        "rìa nam",
        "phía nam",
        "hướng nam",
        "mạn nam",
        "bên nam",
        "đầu nam",
        "cuối nam",
        "ngoài nam",
        "dưới cùng",
        "dưới nhất",
        "thấp nhất",
        "cực dưới",
        "mép dưới",
        "rìa dưới",
        "đáy dưới",
        "tận dưới",
    ],

    "east_most": [
        "đông nhất",
        "cực đông",
        "tận đông",
        "mép đông",
        "rìa đông",
        "phía đông",
        "hướng đông",
        "mạn đông",
        "bên đông",
        "đầu đông",
        "cuối đông",
        "ngoài đông",
        "phải nhất",
        "phải cùng",
        "cực phải",
        "mép phải",
        "rìa phải",
        "bên phải",
        "tận phải",
        "ngoài phải",
    ],

    "west_most": [
        "tây nhất",
        "cực tây",
        "tận tây",
        "mép tây",
        "rìa tây",
        "phía tây",
        "hướng tây",
        "mạn tây",
        "bên tây",
        "đầu tây",
        "cuối tây",
        "ngoài tây",
        "trái nhất",
        "trái cùng",
        "cực trái",
        "mép trái",
        "rìa trái",
        "bên trái",
        "tận trái",
        "ngoài trái",
    ],
}

# Kiểm tra chính xác 20 cụm, mỗi cụm đúng 2 tiếng.
for kind, aliases in MAP_SPATIAL_ALIASES.items():
    assert len(aliases) == 20, (kind, len(aliases))
    assert len(set(aliases)) == 20, f"Trùng alias: {kind}"
    assert all(len(alias.split()) == 2 for alias in aliases), kind

# ============================================================================
# 5. CHUYỂN CỤM TỪ THÀNH MAP_TPL
#
# MAP_TPL phải chứa template mà v4_spatial_sample() dùng trực tiếp.
# Những cụm không có nghĩa cực trị được đặt trong khung có "nhất",
# tránh gán nhầm near -> anchor_near, west -> west_most.
# ============================================================================

SPATIAL_EXTREME_WORD = {
    "north_most": "phía bắc",
    "south_most": "phía nam",
    "east_most": "phía đông",
    "west_most": "phía tây",
}

SPATIAL_EXPLICIT_EXTREME = {
    "north_most": {
        "bắc nhất", "cực bắc", "trên cùng", "cao nhất"
    },
    "south_most": {
        "nam nhất", "cực nam", "dưới cùng", "thấp nhất"
    },
    "east_most": {
        "đông nhất", "cực đông", "phải nhất"
    },
    "west_most": {
        "tây nhất", "cực tây", "trái nhất"
    },
}

def build_spatial_templates(kind, aliases):
    templates = []

    if kind == "anchor_near":
        for alias in aliases:
            if alias in ("gần nhất", "sát nhất"):
                template = f"địa điểm {alias} {{A}}"
            else:
                template = (
                    f"chỗ {alias} {{A}}, "
                    f"chọn nơi gần {{A}} nhất trong tất cả địa điểm"
                )
            templates.append(template)

    else:
        direction = SPATIAL_EXTREME_WORD[kind]
        explicit = SPATIAL_EXPLICIT_EXTREME[kind]

        for alias in aliases:
            if alias in explicit:
                template = f"địa điểm {alias} trên bản đồ"
            else:
                template = (
                    f"địa điểm ở {alias} của bản đồ, "
                    f"chọn nơi xa nhất về {direction}"
                )
            templates.append(template)

    return templates


for kind, aliases in MAP_SPATIAL_ALIASES.items():
    new_templates = build_spatial_templates(kind, aliases)

    # Thay đúng bộ MAP_TPL của 5 loại.
    # Không tác động các biến BANK['near'], BANK['west'], ...
    MAP_TPL[kind] = list(dict.fromkeys(new_templates))

    assert len(MAP_TPL[kind]) == 20, (
        kind, len(MAP_TPL[kind])
    )

# ============================================================================
# 6. ADDITIONAL PLACE ALIASES — GIỮ NGUYÊN
# ============================================================================

ADDITIONAL_SAFE_ALIASES = {
    "canteen": [
        "quầy đồ uống",
        "suất ăn",
        "cơm căn tin",
    ],
    "clinic": [
        "phòng khám bệnh",
        "khu khám bệnh",
        "phòng bác sĩ",
    ],
    "dorm": [
        "ký túc xá",
        "nội trú",
        "khu ký túc",
    ],
    "gate": [
        "lối ra vào",
        "cổng trường",
        "trạm gác",
    ],
    "lab": [
        "phòng thực nghiệm",
        "phòng hóa nghiệm",
        "phòng kiểm nghiệm",
    ],
    "lecture": [
        "giảng đường lớn",
        "giảng đường nhỏ",
        "dãy giảng đường",
    ],
    "library": [
        "phòng đọc sách",
        "phòng thư viện",
        "quầy mượn sách",
    ],
    "office": [
        "tòa hiệu bộ",
        "phòng giáo vụ",
        "văn phòng trường",
    ],
    "parking": [
        "hầm để xe",
        "bãi giữ xe",
        "nhà gửi xe",
    ],
    "sports": [
        "sân bóng chuyền",
        "nhà thi đấu",
        "sân thể thao",
    ],
}

for _kind, _aliases in ADDITIONAL_SAFE_ALIASES.items():
    BANK[_kind] = list(
        dict.fromkeys(BANK[_kind] + _aliases)
    )

if "_ALIAS_CACHE" in globals():
    _ALIAS_CACHE.clear()

if "_CLASH" in globals():
    _CLASH.clear()

# ============================================================================
# 7. VALIDATION
# ============================================================================

print("=" * 65)
print("V3 + SPATIAL ALIASES UPDATED")
print("=" * 65)

total = 0

for kind, aliases in MAP_SPATIAL_ALIASES.items():
    n = len(aliases)
    total += n

    print(
        f"{kind:15s} | "
        f"{n:2d} cụm 2 tiếng | "
        f"{len(MAP_TPL[kind]):2d} templates"
    )

print("-" * 65)
print("TOTAL SPATIAL ALIASES:", total)
print("TOTAL SPATIAL TEMPLATES:", sum(
    len(MAP_TPL[k]) for k in MAP_SPATIAL_ALIASES
))
print("V3 BUCKETS: giữ nguyên")
print("HEADS / H2I: giữ nguyên")
print("Checkpoint: giữ nguyên")
print("Không nhân V3 thêm ×4")
print("=" * 65)

V3 + SPATIAL ALIASES UPDATED
anchor_near     | 20 cụm 2 tiếng | 20 templates
north_most      | 20 cụm 2 tiếng | 20 templates
south_most      | 20 cụm 2 tiếng | 20 templates
east_most       | 20 cụm 2 tiếng | 20 templates
west_most       | 20 cụm 2 tiếng | 20 templates
-----------------------------------------------------------------
TOTAL SPATIAL ALIASES: 100
TOTAL SPATIAL TEMPLATES: 100
V3 BUCKETS: giữ nguyên
HEADS / H2I: giữ nguyên
Checkpoint: giữ nguyên
Không nhân V3 thêm ×4


In [ ]:
# CELL 10 — Preserve V3 eight preconfigured buckets (NO second ×4). Augment other sources only.
# 4 views: ORIGINAL, NO_ACCENTS, EXACTLY_ONE_CHARACTER_TYPO/DELETION,
#          SAME_TYPO+NO_ACCENTS. Typo is applied exactly once per source sentence.
# Do not corrupt semantic operators such as không/chẳng/nhất/near/far/north or punctuation.
FOUR_VIEWS=('co_dau','khong_dau','sai_thieu_1_ky_tu','sai_thieu_1_ky_tu_khong_dau')
SEMANTIC_CRITICAL={
    'không','ko','k','chẳng','chả','đừng','khỏi','chưa','phải','nữa','nhất','gần','xa',
    'bắc','nam','đông','tây','trước','sau','nhưng','thay','đổi','hủy','nhầm',
    'gấp','vội','khẩn','cấp','ngay','cần','vỡ','bể','móp','gãy','fragile',
    'near','far','north','south','east','west','none', 'via','goal'
}
SEMANTIC_CRITICAL_FOLDED={vocab_fold(x) for x in SEMANTIC_CRITICAL}
def one_char_typo(s, seed):
    """
    Tạo đúng 1 lỗi ký tự trong câu.

    Ưu tiên:
      1. Tên địa điểm trong BANK: phòng học, cổng trường, thư viện...
      2. Tên địa điểm đơn lẻ.
      3. Từ ngữ cảnh thông thường.

    Không cố ý sửa các toán tử ngữ nghĩa:
      không, chẳng, đừng, gần, xa, nhất, bắc, nam...
    """
    rng = random.Random(seed)

    critical = globals().get("SEMANTIC_CRITICAL_FOLDED", set())

    def safe_token(word):
        return (
            len(word) >= 4
            and word.isalpha()
            and vocab_fold(word) not in critical
        )

    def mutate_span(start, end):
        segment = s[start:end]

        # Chọn một từ bên trong cụm địa điểm.
        words = [
            m for m in re.finditer(r"[^\W\d_]+", segment, re.UNICODE)
            if safe_token(m.group())
        ]

        if not words:
            return None

        word_match = rng.choice(words)
        word = word_match.group()

        # Drop / replace / swap / duplicate một ký tự.
        op = rng.choice(["drop", "replace", "swap", "duplicate"])

        if op == "drop" and len(word) > 3:
            pos = rng.randrange(len(word))
            new_word = word[:pos] + word[pos + 1:]

        elif op == "replace":
            pos = rng.randrange(len(word))
            new_char = "x" if word[pos].lower() != "x" else "z"
            new_word = word[:pos] + new_char + word[pos + 1:]

        elif op == "swap" and len(word) > 3:
            pos = rng.randrange(len(word) - 1)
            chars = list(word)
            chars[pos], chars[pos + 1] = chars[pos + 1], chars[pos]
            new_word = "".join(chars)

        else:
            pos = rng.randrange(len(word))
            new_word = word[:pos] + word[pos] + word[pos:]

        if new_word == word:
            return None

        a = start + word_match.start()
        b = start + word_match.end()

        return s[:a] + new_word + s[b:]

    # ============================================================
    # 1. ƯU TIÊN TÊN ĐỊA ĐIỂM
    # ============================================================

    entity_matches = []

    places = globals().get("PLACES", [])

    bank = globals().get("BANK", {})

    for place in places:
        aliases = bank.get(place, [place])

        for alias in aliases:
            alias = str(alias).strip()

            if len(alias) < 4:
                continue

            pattern = (
                r"(?<!\w)"
                + re.escape(alias)
                + r"(?!\w)"
            )

            for match in re.finditer(
                pattern,
                s,
                flags=re.IGNORECASE
            ):
                entity_matches.append(
                    (match.start(), match.end(), alias)
                )

    # Loại trùng các vị trí match
    entity_matches = list({
        (a, b): (a, b, alias)
        for a, b, alias in entity_matches
    }.values())

    # Ưu tiên tên địa điểm trong 80% trường hợp có match
    if entity_matches and rng.random() < 0.80:
        rng.shuffle(entity_matches)

        for start, end, alias in entity_matches:
            result = mutate_span(start, end)

            if result is not None:
                return result

    # ============================================================
    # 2. FALLBACK: TỪ NGỮ CẢNH KHÔNG QUAN TRỌNG
    # ============================================================

    candidates = []

    for match in re.finditer(
        r"[^\W\d_]+",
        s,
        flags=re.UNICODE
    ):
        if safe_token(match.group()):
            candidates.append(
                (match.start(), match.end())
            )

    rng.shuffle(candidates)

    for start, end in candidates:
        result = mutate_span(start, end)

        if result is not None:
            return result

    return s

def four_views(row):
    s=row['text']
    seed=int.from_bytes(hashlib.sha256((s+'|v11-view').encode('utf8')).digest()[:8],'big')
    typo=one_char_typo(s,seed)
    views=(s, vocab_fold(s), typo, vocab_fold(typo))
    # On naturally unaccented official missions, clean and noaccent MAY coincide.
    # Never hallucinate accents into the official text to fake a fourth distinct string.
    return [dict(row,text=t,view=v,base_text=s) for v,t in zip(FOUR_VIEWS,views)]

def extend_4(rows):return [r4 for r in rows for r4 in four_views(r)]

# Generator V3 ALREADY generates Acc/NoAcc × Typo/NoTypo × Aff/Neg.
# Never apply four_views to V3 rows a second time.
V3_HOLD_FAMILIES=('note_wrong','old_info','qa_reject')
# ================================================================
# 10. BUILD TRAIN SET (giữ nguyên luồng cũ; 8 bucket × 5000)
# ================================================================
print("Đang sinh V3: 8 bucket × 5.000 = 40.000 mẫu (ĐÃ có dấu/không dấu/typo)...")
# Giữ nguyên 8 bucket/cách in. 5000 mỗi bucket -> 40.000 tổng, không phải 15.000.
v3_buckets, BalancerObj = generate_15k_dataset(seed=42, num_per_bucket=5000,
                                                dedupe=True, exclude_goal_types=V3_HOLD_FAMILIES)
final_15k_items=[]
for bucket_name,bucket_items in v3_buckets.items():
    final_15k_items.extend(bucket_items)
    print(f"  {bucket_name}: {len(bucket_items)}")
print("Đang gộp thêm dữ liệu Train gốc + no-accent...")
train_extended=[]
for it in train_real:
    train_extended.append({'text':it['text'],'y':it['y']})
    train_extended.append({'text':fold(it['text']),'y':it['y']})
final_train_items=train_extended+final_15k_items
print(f"✅ Synthetic: {len(final_15k_items)}")
print(f"✅ Train gốc mở rộng: {len(train_extended)}")
print(f"✅ Tổng train sau gộp: {len(final_train_items)}")
audit_dataset(v3_buckets)
V3_ALL_BUCKETS=tuple(v3_buckets.keys())  # 8 gốc ×5000, KHÔNG bỏ bucket nào
v3_base=[{'text':r['text'],'y':r['y'],'origin':'v3_'+b,
          'view':'v3_native_'+b,'base_text':r['text']}
         for b in V3_ALL_BUCKETS for r in v3_buckets[b]]
# Mỗi hàng V3 là một mẫu cuối cùng; KHÔNG nhân tiếp ×4.
v3_probe_buckets,_=generate_15k_dataset(seed=4243,num_per_bucket=V3_HARD_PER_BUCKET,
                                         dedupe=True,goal_types=V3_HOLD_FAMILIES)
v3_hard_base=[{'text':r['text'],'y':r['y'],'origin':'v3_hard_'+b,
               'view':'v3_native_'+b,'base_text':r['text']}
              for b in V3_ALL_BUCKETS for r in v3_probe_buckets[b]]

map_train_base=v4_spatial_generate(SPATIAL_TRAIN_BASE,'train',SEED+10000)
map_hard_base=v4_spatial_generate(SPATIAL_HARD_BASE,'holdout',SEED+1000000)
anchor_contrast_base=[]
for j in range(ANCHOR_CONTRAST_BASE):
    a,b=random.Random(SEED+j).sample(PLACES,2)
    anchor_contrast_base.extend([
        v4_spatial_sample(SEED+300000+j*31,'train',force_kind='anchor_near',force_anchor=a),
        v4_spatial_sample(SEED+300000+j*31,'train',force_kind='anchor_near',force_anchor=b)
    ])

# Group-split BEFORE augmentation: no synthetic/real validation and no other views leak into train.
train_base=train_real+v3_base+map_train_base+anchor_contrast_base
hard_base=v3_hard_base+map_hard_base
train_keys={vocab_key(r['text']) for r in train_base}
val_keys={vocab_key(r['text']) for r in official_val}
hard_keys={vocab_key(r['text']) for r in hard_base}
assert not (train_keys & val_keys), 'Train/real-validation content overlap'
assert not (train_keys & hard_keys), 'Train/hard-validation content overlap'
assert not (val_keys & hard_keys), 'Real/hard-validation content overlap'

def check_conflict(rows,who):
    seen={};bad=[]
    for row in rows:
        key=vocab_key(row['text'])
        if key in seen and seen[key]!=row['y']:bad.append((row['text'],row['y'],seen[key]))
        seen[key]=row['y']
    assert not bad, (who,len(bad),bad[:2])
check_conflict(train_base,'TRAIN');check_conflict(hard_base,'HARD');check_conflict(official_val,'OFFICIAL_VAL')
# V3 already includes 8 language/noise buckets: never apply extend_4 to V3.
# Official real missions and spatial/contrast generator are augmented separately.
train_rows = (
    extend_4(train_real)
    + v3_base
    + extend_4(map_train_base)
    + extend_4(anchor_contrast_base)
)
hard_val_rows = v3_hard_base + extend_4(map_hard_base)
official_val_rows = extend_4(official_val)
combined_val_rows = official_val_rows + hard_val_rows
assert len(train_rows) == 4*(len(train_real)+len(map_train_base)+len(anchor_contrast_base)) + len(v3_base)
assert len(hard_val_rows) == len(v3_hard_base) + 4*len(map_hard_base)
assert len(official_val_rows) == 4*len(official_val)
assert not {r['base_text'] for r in train_rows} & {r['base_text'] for r in combined_val_rows}
for rows in (train_rows,official_val_rows,hard_val_rows):
    for r in rows:
        assert set(r['y'])==set(HEADS_EXPECTED)
        assert all(0<=r['y'][h]<len(HEADS_EXPECTED[h]) for h in HEADS_EXPECTED)

print('BASE train REAL=',len(train_real),'V3=',len(v3_base),'SPATIAL=',len(map_train_base),
      'ANCHOR-CONTRAST=',len(anchor_contrast_base))
print('TRAIN total (V3 native + other sources ×4):',len(train_rows))
print('VAL official ×4:',len(official_val_rows),'VAL hard mixed:',len(hard_val_rows),
      'VAL COMBINED ×4:',len(combined_val_rows))
print('TRAIN view distribution:',dict(Counter(r['view'] for r in train_rows)))
print('VAL view distribution:',dict(Counter(r['view'] for r in combined_val_rows)))
print('EXAMPLE 4 views of one original mission:')
for row in four_views(train_real[0]): print('  ',row['view'],':',row['text'][:170])


Đang sinh V3: 8 bucket × 5.000 = 40.000 mẫu (ĐÃ có dấu/không dấu/typo)...
  Aff_Acc_NoTypo: 5000
  Aff_Acc_Typo: 5000
  Aff_NoAcc_NoTypo: 5000
  Aff_NoAcc_Typo: 5000
  Neg_Acc_NoTypo: 5000
  Neg_Acc_Typo: 5000
  Neg_NoAcc_NoTypo: 5000
  Neg_NoAcc_Typo: 5000
Đang gộp thêm dữ liệu Train gốc + no-accent...
✅ Synthetic: 40000
✅ Train gốc mở rộng: 4000
✅ Tổng train sau gộp: 44000
Tổng mẫu: 40000 | skeleton khác nhau: 40000 (100.0%)
Skeleton phổ biến nhất chiếm: 0.003%
Phân bố kiểu goal-unit: {'dispatch_update': 1122, 'anchor_vs_goal': 1335, 'last_minute_update': 1195, 'two_destinations': 839, 'unavailable': 1400, 'other_order': 1395, 'quoted_wrong': 826, 'plain': 6657, 'literal_vs_metaphor': 664, 'self_correct': 1855, 'recipient_vs_sender': 1174, 'preference_override': 1433, 'wrong_recipient': 1128, 'fix_after_cancel': 1804, 'recipient_override': 1067, 'corrected_transcript': 2198, 'reject_multi': 3018, 'reject_goal': 6096, 'cancel_and_replace': 2623, 'denied_hint': 2171}
Goal đứng TRƯỚC di

In [ ]:
# ============================================================================
# CELL 7 — LOAD PHASE 11 CHECKPOINT FOR PHASE 12 TRAINING
# ============================================================================

# ============================================================================
# 0. IMPORTS + CONFIG — KHAI BÁO TRỰC TIẾP NGAY ĐẦU CELL
# ============================================================================

import os
import copy
import gc

from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoModel, AutoTokenizer


# ĐƯỜNG DẪN CHECKPOINT PHASE 11 CHÍNH XÁC
CHECKPOINT_PATH = Path("/content/drive/MyDrive/bamibert_nlu_phase12_v3_4views/bamibert_nlu_phase11_v3_4views_best.pt")

CKPT_PATH = CHECKPOINT_PATH

# Mô hình BamiBERT gốc
MODEL_NAME = "Qualcomm-AI-Research/BamiBERT"

# Thiết lập thiết bị
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

MAX_LEN = 256
EXTRA_POOLS = 3


# ============================================================================
# 1. VERIFY CHECKPOINT PATH
# ============================================================================

if not CKPT_PATH.is_file():

    raise FileNotFoundError(
        f"Không tìm thấy checkpoint Phase 11 tại:\n{CKPT_PATH}\n"
        "Kiểm tra Google Drive đã được mount hay chưa."
    )

if CKPT_PATH.suffix.lower() != ".pt":

    raise ValueError(
        f"Checkpoint phải có đuôi .pt: {CKPT_PATH}"
    )

print("=" * 75)
print("LOADING PHASE 11 CHECKPOINT")
print("SOURCE:", CKPT_PATH)
print("DEVICE:", DEVICE)
print("=" * 75)


# ============================================================================
# 2. LOAD CHECKPOINT
# ============================================================================

checkpoint = torch.load(
    str(CKPT_PATH),
    map_location="cpu",
    weights_only=False
)

assert isinstance(checkpoint, dict), (
    "Checkpoint phải là dictionary"
)

assert "state" in checkpoint, (
    "Checkpoint thiếu key 'state'"
)

assert "heads" in checkpoint, (
    "Checkpoint thiếu key 'heads'"
)

assert checkpoint.get("phase") == 11, (
    "Expected Phase 11",
    checkpoint.get("phase")
)

# Kiểm tra chính xác thứ tự nhãn nếu đã định nghĩa ở các CELL trước
if "HEADS_EXPECTED" in globals():

    assert checkpoint["heads"] == HEADS_EXPECTED, (
        "EXACT CLASS ORDER MISMATCH",
        checkpoint["heads"],
        HEADS_EXPECTED
    )

HEADS = copy.deepcopy(checkpoint["heads"])

assert len(HEADS) == 8, (
    "Expected exactly 8 heads",
    list(HEADS.keys())
)

H2I = {
    head: {
        label: index
        for index, label in enumerate(labels)
    }
    for head, labels in HEADS.items()
}

print("Checkpoint phase:", checkpoint.get("phase"))
print("Checkpoint epoch:", checkpoint.get("epoch"))
print("Number of heads:", len(HEADS))

for head, labels in HEADS.items():
    print(f"  {head}: {len(labels)} labels")


# ============================================================================
# 3. DEFINE MULTI-HEAD NLU
# ============================================================================

class MultiHeadNLU(nn.Module):

    def __init__(self, name, extra_pools=3):
        super().__init__()

        self.enc = AutoModel.from_pretrained(name)

        hidden = self.enc.config.hidden_size

        self.extra_pools = extra_pools
        self.drop = nn.Dropout(0.2)

        # Original attention pooling
        self.pool_q = nn.ModuleDict({
            head: nn.Sequential(
                nn.Linear(hidden, hidden // 2),
                nn.Tanh(),
                nn.Linear(hidden // 2, 1)
            )
            for head in HEADS
        })

        # Original classification heads
        self.heads = nn.ModuleDict({
            head: nn.Linear(
                hidden * 2,
                len(labels)
            )
            for head, labels in HEADS.items()
        })

        # Extra attention pooling
        self.extra_pool_q = nn.ModuleDict({
            head: nn.Linear(
                hidden,
                extra_pools,
                bias=False
            )
            for head in HEADS
        })

        extra_size = hidden * (extra_pools + 1)

        # Extra normalization
        self.extra_norm = nn.ModuleDict({
            head: nn.LayerNorm(extra_size)
            for head in HEADS
        })

        # Extra classification heads
        self.extra_heads = nn.ModuleDict({
            head: nn.Linear(
                extra_size,
                len(labels)
            )
            for head, labels in HEADS.items()
        })

        # Weight initialization
        for head in HEADS:

            nn.init.zeros_(
                self.pool_q[head][2].weight
            )

            nn.init.zeros_(
                self.pool_q[head][2].bias
            )

            nn.init.normal_(
                self.extra_pool_q[head].weight,
                mean=0.0,
                std=0.02
            )

            nn.init.zeros_(
                self.extra_heads[head].weight
            )

            nn.init.zeros_(
                self.extra_heads[head].bias
            )

    # ========================================================================
    # FORWARD
    # ========================================================================

    def forward(self, ids, mask, return_emb=False):

        hidden = self.enc(
            input_ids=ids,
            attention_mask=mask
        ).last_hidden_state

        cls = hidden[:, 0, :]

        logits = {}

        token_mask = mask.unsqueeze(-1) == 0

        for head in HEADS:

            # Original attention weights
            weight = (
                self.pool_q[head](hidden)
                .squeeze(-1)
                .float()
                .masked_fill(mask == 0, -1e4)
                .softmax(-1)
                .to(hidden.dtype)
            )

            pooled = (
                weight.unsqueeze(-1) * hidden
            ).sum(dim=1)

            old_logits = self.heads[head](
                self.drop(
                    torch.cat(
                        [cls, pooled],
                        dim=-1
                    )
                )
            )

            # Extra multi-pool attention
            extra_score = (
                self.extra_pool_q[head](hidden)
                .float()
                .masked_fill(token_mask, -1e4)
            )

            extra_weight = (
                extra_score.softmax(dim=1)
                .to(hidden.dtype)
            )

            extra_pool = torch.einsum(
                "btk,bth->bkh",
                extra_weight,
                hidden
            ).flatten(1)

            extra_feature = self.extra_norm[head](
                torch.cat(
                    [cls, extra_pool],
                    dim=-1
                )
            )

            # Final logits
            logits[head] = (
                old_logits
                + self.extra_heads[head](
                    self.drop(extra_feature)
                )
            )

        if not return_emb:
            return logits

        # Normalized sentence embedding
        mask3 = mask.unsqueeze(-1).to(hidden.dtype)

        mean = (
            (hidden * mask3).sum(dim=1)
            / mask3.sum(dim=1).clamp(min=1)
        )

        emb = F.normalize(
            F.normalize(
                cls.float(),
                dim=-1
            )
            + F.normalize(
                mean.float(),
                dim=-1
            ),
            dim=-1
        )

        return logits, emb


# ============================================================================
# 4. LOAD TOKENIZER
# ============================================================================

print("\nLoading tokenizer...")

tok = AutoTokenizer.from_pretrained(
    MODEL_NAME
)


# ============================================================================
# 5. CREATE MODEL
# ============================================================================

print("Creating MultiHeadNLU...")

model = MultiHeadNLU(
    MODEL_NAME,
    extra_pools=EXTRA_POOLS
)


# ============================================================================
# 6. LOAD PHASE 11 WEIGHTS — STRICT
# ============================================================================

print("\nLoading Phase 11 weights with strict=True...")

load_result = model.load_state_dict(
    checkpoint["state"],
    strict=True
)

assert not load_result.missing_keys
assert not load_result.unexpected_keys

model = model.to(DEVICE)

print("STRICT WEIGHTS LOADED SUCCESSFULLY")


# ============================================================================
# 7. RELEASE CHECKPOINT MEMORY
# ============================================================================

del checkpoint

gc.collect()

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


# ============================================================================
# 8. VERIFY MODEL FORWARD
# ============================================================================

model.eval()

test_text = (
    "Giao hàng đến thư viện, "
    "ghé căn tin lấy hàng trước, "
    "đơn này gấp và dễ vỡ."
)

inputs = tok(
    test_text,
    return_tensors="pt",
    truncation=True,
    max_length=MAX_LEN,
    padding=True
)

test_ids = inputs["input_ids"].to(DEVICE)
test_mask = inputs["attention_mask"].to(DEVICE)

with torch.inference_mode():

    test_logits, test_emb = model(
        test_ids,
        test_mask,
        return_emb=True
    )

# Kiểm tra đầy đủ 8 heads
assert set(test_logits.keys()) == set(HEADS.keys())

for head, labels in HEADS.items():

    expected_shape = (1, len(labels))

    actual_shape = tuple(
        test_logits[head].shape
    )

    assert actual_shape == expected_shape, (
        head,
        actual_shape,
        expected_shape
    )

    assert torch.isfinite(
        test_logits[head]
    ).all(), f"Non-finite logits: {head}"

assert torch.isfinite(test_emb).all(), (
    "Non-finite sentence embedding"
)

# Chuyển model sang training mode
model.train()


# ============================================================================
# 9. FINAL SUMMARY
# ============================================================================

print("\n" + "=" * 75)
print("PHASE 11 CHECKPOINT LOADED SUCCESSFULLY")
print("=" * 75)

print("SOURCE:", CKPT_PATH)
print("MODEL:", MODEL_NAME)
print("DEVICE:", DEVICE)
print("HEADS:", {
    head: len(labels)
    for head, labels in HEADS.items()
})
print("EMBEDDING SHAPE:", tuple(test_emb.shape))
print("STRICT STATE_DICT: PASSED")
print("FORWARD TEST: PASSED")
print("MODEL TRAIN MODE:", model.training)
print("READY FOR PHASE 12 TRAINING")

print("=" * 75)

LOADING PHASE 11 CHECKPOINT
SOURCE: /content/drive/MyDrive/bamibert_nlu_phase12_v3_4views/bamibert_nlu_phase11_v3_4views_best.pt
DEVICE: cuda
Checkpoint phase: 11
Checkpoint epoch: 1
Number of heads: 8
  goal: 11 labels
  via: 11 labels
  gkind: 12 labels
  ganchor: 11 labels
  vkind: 12 labels
  vanchor: 11 labels
  urgent: 2 labels
  fragile: 2 labels

Loading tokenizer...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Creating MultiHeadNLU...


config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/412M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at Qualcomm-AI-Research/BamiBERT and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Loading Phase 11 weights with strict=True...
STRICT WEIGHTS LOADED SUCCESSFULLY

PHASE 11 CHECKPOINT LOADED SUCCESSFULLY
SOURCE: /content/drive/MyDrive/bamibert_nlu_phase12_v3_4views/bamibert_nlu_phase11_v3_4views_best.pt
MODEL: Qualcomm-AI-Research/BamiBERT
DEVICE: cuda
HEADS: {'goal': 11, 'via': 11, 'gkind': 12, 'ganchor': 11, 'vkind': 12, 'vanchor': 11, 'urgent': 2, 'fragile': 2}
EMBEDDING SHAPE: (1, 768)
STRICT STATE_DICT: PASSED
FORWARD TEST: PASSED
MODEL TRAIN MODE: True
READY FOR PHASE 12 TRAINING


In [ ]:
# 8. Model data loaders + RAW and CONSTRAINED accuracies.
# No synthetic/official validation examples enter the optimizer.
class NLUDataset(Dataset):
    def __init__(self,rows):self.rows=rows
    def __len__(self):return len(self.rows)
    def __getitem__(self,j):
        r=self.rows[j]
        return r['text'],r['y']

def collate(batch):
    enc=tok([t for t,y in batch],padding=True,truncation=True,max_length=MAX_LEN,
            return_tensors='pt')
    ys={h:torch.tensor([y[h] for _,y in batch],dtype=torch.long) for h in HEADS}
    return enc,ys

def loader(rows,batch,shuffle=False):
    return DataLoader(NLUDataset(rows),batch_size=batch,shuffle=shuffle,collate_fn=collate,
                      num_workers=0,pin_memory=(DEVICE.type=='cuda'))

# Reuse the ORIGINAL Phase8 constrained decoding (keeps both selector label and anchor valid).
P8_PLACES=PLACES
P8_MAP=MAP_KINDS
P8_DIRECT=set(DIRECT_KINDS)
P8_ANCHORED={'near','far','anchor_near'}
def p8_decode(logs):
    """Constrained MAP decoder: no-via is distinct from map-dependent via=none."""
    n=len(logs['goal']); pr={h:np.zeros(n,dtype=np.int64) for h in HEADS}
    for h in ('urgent','fragile'):pr[h]=np.argmax(logs[h],axis=1)
    def choose_ref(i,kind_head,anchor_head,self_place,kinds):
        best=(-1e90,None,None)
        for kind in kinds:
            ki=H2I[kind_head][kind]
            if kind in P8_ANCHORED:
                candidates=[H2I[anchor_head][p] for p in P8_PLACES
                            if kind=='anchor_near' or p!=self_place]
                ai=max(candidates,key=lambda x:logs[anchor_head][i,x])
            else:ai=H2I[anchor_head]['none']
            sc=float(logs[kind_head][i,ki]+logs[anchor_head][i,ai])
            if sc>best[0]:best=(sc,ki,ai)
        return best
    for i in range(n):
        goals=[];vias=[]
        for gi,g in enumerate(HEADS['goal']):
            sc,k,a=choose_ref(i,'gkind','ganchor',g,P8_MAP if g=='none' else P8_DIRECT)
            goals.append((float(logs['goal'][i,gi])+sc,gi,k,a))
        for vi,v in enumerate(HEADS['via']):
            if v=='none':
                ki=H2I['vkind']['none'];ai=H2I['vanchor']['none']
                no=(float(logs['via'][i,vi]+logs['vkind'][i,ki]+logs['vanchor'][i,ai]),vi,ki,ai)
                sc,k,a=choose_ref(i,'vkind','vanchor','none',P8_MAP)
                mp=(float(logs['via'][i,vi])+sc,vi,k,a)
                vias.append(max(no,mp,key=lambda x:x[0]))
            else:
                sc,k,a=choose_ref(i,'vkind','vanchor',v,P8_DIRECT)
                vias.append((float(logs['via'][i,vi])+sc,vi,k,a))
        possibilities=[(g[0]+v[0],g,v) for g in goals for v in vias
                       if not (HEADS['goal'][g[1]]!='none' and HEADS['goal'][g[1]]==HEADS['via'][v[1]])]
        _,g,v=max(possibilities,key=lambda x:x[0])
        pr['goal'][i],pr['gkind'][i],pr['ganchor'][i]=g[1:]
        pr['via'][i],pr['vkind'][i],pr['vanchor'][i]=v[1:]
    return pr



@torch.inference_mode()
def evaluate_rows(rows, description='', print_views=False):
    model.eval(); records=[]
    dl=loader(rows,EVAL_BATCH)
    offset=0
    for enc,ys in dl:
        bs=len(ys['goal'])
        enc={k:v.to(DEVICE,non_blocking=True) for k,v in enc.items()}
        logs=model(enc['input_ids'],enc['attention_mask'])
        scores={h:x.float().log_softmax(-1).cpu().numpy() for h,x in logs.items()}
        raw={h:np.argmax(s,axis=1) for h,s in scores.items()}
        con=p8_decode(scores)
        gt={h:y.numpy() for h,y in ys.items()}
        for i in range(bs):
            rr=rows[offset+i]
            records.append({'view':rr['view'], 'origin':rr.get('origin',''),
                            'raw_exact8':all(int(raw[h][i])==int(gt[h][i]) for h in HEADS),
                            'constrained_exact8':all(int(con[h][i])==int(gt[h][i]) for h in HEADS),
                            'heads':{h:int(con[h][i])==int(gt[h][i]) for h in HEADS}})
        offset+=bs
    assert offset==len(rows)
    n=len(records)
    result={'n':n,
      'raw_exact8':sum(r['raw_exact8'] for r in records)/n,
      'constrained_exact8':sum(r['constrained_exact8'] for r in records)/n,
      'heads':{h:sum(r['heads'][h] for r in records)/n for h in HEADS}}
    for view in FOUR_VIEWS:
        xs=[r for r in records if r['view']==view]
        result[view]={'n':len(xs),'raw_exact8':sum(r['raw_exact8'] for r in xs)/max(len(xs),1),
                      'constrained_exact8':sum(r['constrained_exact8'] for r in xs)/max(len(xs),1)}
    if print_views:
        print('====',description,':',n,'rows | exact8 constrained:',round(result['constrained_exact8'],5))
        for v in FOUR_VIEWS:
            print(' ',v, 'n=',result[v]['n'],'raw=',round(result[v]['raw_exact8'],5),
                  'constrained=',round(result[v]['constrained_exact8'],5))
        print(' per-head:',{h:round(v,5) for h,v in result['heads'].items()})
    return result

def evaluate_all(print_views=True):
    real=evaluate_rows(official_val_rows,'REAL OFFICIAL',print_views)
    hard=evaluate_rows(hard_val_rows,'SYNTHETIC HARD',print_views)
    combined=evaluate_rows(combined_val_rows,'COMBINED REAL + HARD',print_views)
    # Selection metric uses balanced real/hard (not micro-average dominated by generated rows).
    score=W_VAL*real['constrained_exact8']+W_PROBE*hard['constrained_exact8']
    print('balanced 0.6 REAL + 0.4 HARD =',round(score,5)) if print_views else None
    return score,real,hard,combined

baseline_score,base_official,base_hard,base_combined=evaluate_all(print_views=True)


==== REAL OFFICIAL : 1200 rows | exact8 constrained: 1.0
  co_dau n= 300 raw= 1.0 constrained= 1.0
  khong_dau n= 300 raw= 1.0 constrained= 1.0
  sai_thieu_1_ky_tu n= 300 raw= 1.0 constrained= 1.0
  sai_thieu_1_ky_tu_khong_dau n= 300 raw= 1.0 constrained= 1.0
 per-head: {'goal': 1.0, 'via': 1.0, 'gkind': 1.0, 'ganchor': 1.0, 'vkind': 1.0, 'vanchor': 1.0, 'urgent': 1.0, 'fragile': 1.0}
==== SYNTHETIC HARD : 5600 rows | exact8 constrained: 0.98286
  co_dau n= 900 raw= 1.0 constrained= 1.0
  khong_dau n= 900 raw= 0.94889 constrained= 0.94889
  sai_thieu_1_ky_tu n= 900 raw= 1.0 constrained= 1.0
  sai_thieu_1_ky_tu_khong_dau n= 900 raw= 0.94778 constrained= 0.94778
 per-head: {'goal': 0.99982, 'via': 1.0, 'gkind': 0.98339, 'ganchor': 0.99982, 'vkind': 1.0, 'vanchor': 1.0, 'urgent': 0.99982, 'fragile': 0.99982}
==== COMBINED REAL + HARD : 6800 rows | exact8 constrained: 0.98588
  co_dau n= 1200 raw= 1.0 constrained= 1.0
  khong_dau n= 1200 raw= 0.96167 constrained= 0.96167
  sai_thieu_1_ky_t

In [ ]:
# ============================================================================
# PHASE 11 / FULL COLAB — P5 STYLE FINETUNE, LOADED FROM PHASE10 CHECKPOINT
# Same two-tier AdamW, mixed precision, alignment pairs, EMA, clipping,
# cosine alignment, warmup, best-epoch guard as original P5 flow.
# Official validation is NOT used to select checkpoints (avoid overfitting).
# ============================================================================
import os, gc, json, copy, math, random
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import get_linear_schedule_with_warmup
from tqdm.auto import tqdm

LR_ENC_P5, LR_HEAD_P5 = 2e-6, 1.8e-5
P5_EPOCHS, P5_PATIENCE = 3, globals().get("P5_PATIENCE",5)
NOISE_HI_P5, NOISE_LO_P5 = 0.5, 0.3
W_VAL, W_PROBE = 0.6, 0.4
VAL_MAX_DROP = 0.01
ALIGN_W = 0.07
EMA_DECAY = 0.995
WD = 0.01
# Can turn this down if the GPU/runtime budget is limited.
USE_AMP = DEVICE.type=='cuda'
AMP_DTYPE = torch.bfloat16 if USE_AMP and torch.cuda.is_bf16_supported() else torch.float16
USE_SCALER = USE_AMP and AMP_DTYPE==torch.float16

# Keep the pre-trained Phase10 MultiHeadNLU; don't change its head definitions.
# add pairs only as views of the SAME labeled example, never as new semantic labels.
class TrainDS(Dataset):
    def __init__(self, rows):
        self.rows = rows
        self.noise_scale = NOISE_HI_P5
        self.rng = random.Random(SEED+177)

    def __len__(self): return len(self.rows)

    def __getitem__(self,j):
        item=self.rows[j]
        original=item['text']
        # No-accent paired view is always label invariant.
        pair=vocab_fold(original) if self.rng.random()<self.noise_scale else None
        if pair==original: pair=None
        return original,item['y'],pair

def collate_p5(batch):
    texts=[row[0] for row in batch]
    ids=tok(texts,padding=True,truncation=True,max_length=MAX_LEN,return_tensors='pt')
    ys={h:torch.tensor([row[1][h] for row in batch],dtype=torch.long) for h in HEADS}
    pair_indices=[i for i,r in enumerate(batch) if r[2] is not None]
    if pair_indices:
        pair=tok([batch[i][2] for i in pair_indices],padding=True,truncation=True,
                 max_length=MAX_LEN,return_tensors='pt')
        pidx=torch.tensor(pair_indices,dtype=torch.long)
        ib,mb=pair['input_ids'],pair['attention_mask']
    else:
        pidx=ib=mb=None
    return ids['input_ids'],ids['attention_mask'],ys,pidx,ib,mb

WEIGHTS={'goal':1.8,'via':1.8,'gkind':2.1,'ganchor':1.9,
         'vkind':1.7,'vanchor':1.6,'urgent':1.2,'fragile':1.0}
W_SUM=sum(WEIGHTS.values())
def compute_loss(logits, y):
    return sum(WEIGHTS[h]*F.cross_entropy(logits[h].float(),y[h].to(DEVICE))
               for h in HEADS)/W_SUM

class EMA:
    def __init__(self,model,decay):
        self.decay=decay
        self.shadow={k:p.detach().clone() for k,p in model.named_parameters() if p.requires_grad}
        self.backup={}

    @torch.no_grad()
    def update(self,model):
        for k,p in model.named_parameters():
            if p.requires_grad:self.shadow[k].mul_(self.decay).add_(p.detach(),alpha=1-self.decay)

    @torch.no_grad()
    def apply(self,model):
        self.backup={}
        for k,p in model.named_parameters():
            if p.requires_grad:
                self.backup[k]=p.detach().clone()
                p.copy_(self.shadow[k])

    @torch.no_grad()
    def restore(self,model):
        for k,p in model.named_parameters():
            if k in self.backup:p.copy_(self.backup[k])
        self.backup.clear()

def p4_score(model=None):
    # Evaluate FOUR views on REAL official validation AND generated HARD validation each epoch.
    score,sv,sp,sc=evaluate_all(print_views=False)
    return score, sv['constrained_exact8'], sp['constrained_exact8'], sv, sp

# Same structure as the given P5 cell, but use actual Phase10 weights and
# keep the official validation entirely OUTSIDE checkpoint selection.
gc.collect()
if USE_AMP:torch.cuda.empty_cache()
random.seed(SEED);torch.manual_seed(SEED)
ds=TrainDS(train_rows)
loader_p5=DataLoader(ds,batch_size=BATCH_SIZE,shuffle=True,collate_fn=collate_p5,num_workers=0)
print(f'mẫu / epoch: {len(ds)} | bước / epoch: {len(loader_p5)}')
enc_p=[p for k,p in model.named_parameters() if k.startswith('enc.')]
head_p=[p for k,p in model.named_parameters() if not k.startswith('enc.')]
opt=torch.optim.AdamW([{'params':enc_p,'lr':LR_ENC_P5},
                       {'params':head_p,'lr':LR_HEAD_P5}],weight_decay=WD)
total=len(loader_p5)*P5_EPOCHS
sched=get_linear_schedule_with_warmup(opt,int(.1*total),total)
scaler=torch.amp.GradScaler('cuda',enabled=USE_SCALER)
ema=EMA(model,EMA_DECAY)
SAVE_PATH=OUT_DIR/OUT_NAME

def save_checkpoint(epoch,sv,sp):
    torch.save({'phase':11,'subphase':'v3_extended_p5_style',
                'epoch':epoch,'heads':copy.deepcopy(HEADS),
                'state':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()},
                'source_checkpoint':str(CKPT_PATH),
                'real_train':len(train_real),'train_rows':len(train_rows),
                'v3_ood_probes':len(v3_hard_base),
                'official_val':sv,'hard_probe':sp,
                'config':{'encoder_lr':LR_ENC_P5,'head_lr':LR_HEAD_P5,
                          'ema_decay':EMA_DECAY,'align_w':ALIGN_W,
                          'max_epochs':P5_EPOCHS,
                          'wiki_seed_per_group':20,
                          'wiki_auto_import':False}},SAVE_PATH)

base_sc,base_sv,base_sp,_,rp0=p4_score(model)
print(f'Điểm xuất phát (ft5): tổng={base_sc:.4f} | val={base_sv:.4f} | probe={base_sp:.4f}')
best,best_ep,bad=base_sc,0,0
# Valid Phase10 fallback: saved before any optimization.
save_checkpoint(0,base_official,base_hard)
history=[]

for ep in range(1,P5_EPOCHS+1):
    ds.noise_scale=NOISE_HI_P5 if ep<=0.6*P5_EPOCHS else NOISE_LO_P5
    model.train();run=run_al=0.0
    for ids,mask,y,pidx,ib,mb in tqdm(loader_p5,desc=f'P5-{ep}',leave=False):
        opt.zero_grad(set_to_none=True)
        with torch.autocast(DEVICE.type,dtype=AMP_DTYPE,enabled=USE_AMP):
            lg_a,ea=model(ids.to(DEVICE),mask.to(DEVICE),return_emb=True)
            loss=compute_loss(lg_a,y)
            al=torch.zeros((),device=DEVICE)
            if pidx is not None:
                lg_b,eb=model(ib.to(DEVICE),mb.to(DEVICE),return_emb=True)
                loss=loss+compute_loss(lg_b,{h:y[h][pidx] for h in HEADS})
                al=(1-(ea[pidx.to(DEVICE)]*eb).sum(-1)).mean()
                loss=loss+ALIGN_W*al
        scaler.scale(loss).backward();scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        if USE_SCALER:
            sb=scaler.get_scale();scaler.step(opt);scaler.update()
            if scaler.get_scale()>=sb:sched.step()
        else:
            opt.step();sched.step()
        ema.update(model)
        run+=float(loss.detach());run_al+=float(al.detach())
    # Evaluate EMA weights; save only a genuinely improved checkpoint.
    ema.apply(model)
    sc,sv,sp,rv,rp=p4_score(model)
    msg=(f'P5-{ep:02d} loss={run/len(loader_p5):.3f} align={run_al/len(loader_p5):.3f} | tổng={sc:.4f} val={sv:.4f} probe={sp:.4f} | '
         + 'val: '+' '.join(f"{n}:{rv[n]['constrained_exact8']:.3f}" for n in FOUR_VIEWS)
         + ' | probe: '+' '.join(f"{n}:{rp[n]['constrained_exact8']:.3f}" for n in FOUR_VIEWS))
    accepted=sc>best+1e-4 and sv>=base_sv-VAL_MAX_DROP
    if accepted:
        best,best_ep,bad=sc,ep,0
        save_checkpoint(ep,rv,rp)
        msg+=' * (kỷ lục mới)'
    else: bad+=1
    history.append({'epoch':ep,'score':sc,'holdout':sv,'probe':sp,'loss':run/len(loader_p5),
                    'accepted':accepted})
    ema.restore(model);print(msg)
    if bad>=P5_PATIENCE:
        print('Dừng sớm.');break

best_ck=torch.load(SAVE_PATH,map_location='cpu',weights_only=False)
assert best_ck['heads']==HEADS and best_ck['phase']==11
model.load_state_dict(best_ck['state'],strict=True)
model.to(DEVICE)
final_score,final_official,final_hard,final_combined=evaluate_all(print_views=True)
print(f'HOÀN TẤT. Best epoch={best_ep} balanced={best:.4f}, saved: {SAVE_PATH}')
print('Official validation baseline:',base_official)
print('Official validation final   :',final_official)
print('Generated HARD baseline    :',base_hard)
print('Generated HARD final       :',final_hard)
print('COMBINED baseline          :',base_combined)
print('COMBINED final             :',final_combined)
report={'checkpoint':str(SAVE_PATH),'source_checkpoint':str(CKPT_PATH),
        'best_epoch':best_ep,'best_score':best,'final_score':final_score,
        'baseline_official':base_official,'baseline_hard':base_hard,
        'baseline_combined':base_combined,'final_official':final_official,
        'final_hard':final_hard,'final_combined':final_combined,
        'history':history,'wiki_seed_count':{k:len(v) for k,v in WIKI_SEEDS.items()},
        'real_train_base':len(train_real),'v3_base':len(v3_base),
        'spatial_base':len(map_train_base),'train_views_total':len(train_rows),
        'official_val_views_total':len(official_val_rows),
        'hard_val_views_total':len(hard_val_rows),
        'views':list(FOUR_VIEWS),
        'dataset_train_path':TRAIN_SCENES,'dataset_validation_path':VAL_SCENES,
        'selection':'0.6 REAL official val + 0.4 generated OOD hard val; neither used in optimizer'}
(OUT_DIR/'phase11_v3_4views_report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf8')


mẫu / epoch: 83200 | bước / epoch: 2600
Điểm xuất phát (ft5): tổng=0.9931 | val=1.0000 | probe=0.9829


P5-1:   0%|          | 0/2600 [00:00<?, ?it/s]

P5-01 loss=0.004 align=0.002 | tổng=0.9998 val=1.0000 probe=0.9995 | val: co_dau:1.000 khong_dau:1.000 sai_thieu_1_ky_tu:1.000 sai_thieu_1_ky_tu_khong_dau:1.000 | probe: co_dau:1.000 khong_dau:1.000 sai_thieu_1_ky_tu:1.000 sai_thieu_1_ky_tu_khong_dau:1.000 * (kỷ lục mới)


P5-2:   0%|          | 0/2600 [00:00<?, ?it/s]

P5-02 loss=0.003 align=0.002 | tổng=0.9998 val=1.0000 probe=0.9995 | val: co_dau:1.000 khong_dau:1.000 sai_thieu_1_ky_tu:1.000 sai_thieu_1_ky_tu_khong_dau:1.000 | probe: co_dau:1.000 khong_dau:1.000 sai_thieu_1_ky_tu:1.000 sai_thieu_1_ky_tu_khong_dau:1.000


P5-3:   0%|          | 0/2600 [00:00<?, ?it/s]

P5-03 loss=0.003 align=0.001 | tổng=0.9997 val=1.0000 probe=0.9993 | val: co_dau:1.000 khong_dau:1.000 sai_thieu_1_ky_tu:1.000 sai_thieu_1_ky_tu_khong_dau:1.000 | probe: co_dau:1.000 khong_dau:1.000 sai_thieu_1_ky_tu:1.000 sai_thieu_1_ky_tu_khong_dau:1.000
==== REAL OFFICIAL : 1200 rows | exact8 constrained: 1.0
  co_dau n= 300 raw= 1.0 constrained= 1.0
  khong_dau n= 300 raw= 1.0 constrained= 1.0
  sai_thieu_1_ky_tu n= 300 raw= 1.0 constrained= 1.0
  sai_thieu_1_ky_tu_khong_dau n= 300 raw= 1.0 constrained= 1.0
 per-head: {'goal': 1.0, 'via': 1.0, 'gkind': 1.0, 'ganchor': 1.0, 'vkind': 1.0, 'vanchor': 1.0, 'urgent': 1.0, 'fragile': 1.0}
==== SYNTHETIC HARD : 5600 rows | exact8 constrained: 0.99946
  co_dau n= 900 raw= 1.0 constrained= 1.0
  khong_dau n= 900 raw= 1.0 constrained= 1.0
  sai_thieu_1_ky_tu n= 900 raw= 1.0 constrained= 1.0
  sai_thieu_1_ky_tu_khong_dau n= 900 raw= 1.0 constrained= 1.0
 per-head: {'goal': 0.99982, 'via': 1.0, 'gkind': 1.0, 'ganchor': 0.99982, 'vkind': 1.0, '

6563

In [ ]:
# CELL 14 — Audits: four versions and explicit train/validation separation
print('OFFICIAL real val (4 views):',len(official_val_rows))
print('GENERATED hard val (V3 native + spatial views):',len(hard_val_rows))
print('COMBINED val:',len(combined_val_rows))
print('TRAIN (V3 native + other views):',len(train_rows))
print('Best checkpoint:',SAVE_PATH)
print('Per-view scores in:',OUT_DIR/'phase11_v3_4views_report.json')
for src in (train_real[0],official_val[0]):
    print('\nORIGIN:',src['origin'])
    for r in four_views(src): print(' ',r['view'],'=>',r['text'][:180])


OFFICIAL real val (4 views): 1200
GENERATED hard val (V3 native + spatial views): 5600
COMBINED val: 6800
TRAIN (V3 native + other views): 83200
Best checkpoint: /content/drive/MyDrive/bamibert_nlu_phase12_v3_4views/bamibert_nlu_phase11_v3_4views_best.pt
Per-view scores in: /content/drive/MyDrive/bamibert_nlu_phase12_v3_4views/phase11_v3_4views_report.json

ORIGIN: official_train
  co_dau => Xin chao! Dung hnam voi phong doc nhe. Huy don giao nha xe. Thay vao do: Truoc khi chuyen bien ten khach den khong phai nha thi dau ma la cong truong gip minh, nho ghe lop hoc lay 
  khong_dau => xin chao! dung hnam voi phong doc nhe. huy don giao nha xe. thay vao do: truoc khi chuyen bien ten khach den khong phai nha thi dau ma la cong truong gip minh, nho ghe lop hoc lay 
  sai_thieu_1_ky_tu => Xin chao! Dung hnam voi phong doc nhe. Huy don giao nha xe. Thay vao do: Truoc khi chuyen bien ten khach den khong phai nha thi dau ma la cong truong gip minh, nho ghe lop hoc lay 
  sai_thieu_1_ky_tu_khon

Notebook được lưu thành checkpoint Phase 11 và report trên Google Drive. Phần Wiki tự chạy theo cấu hình gốc của bạn; không lược bỏ hàm cào hay lệnh print.
